# PHOEBE Contact Binary Lab

An experimental **contact-binary** companion to PHOEBE Lab. The instant preview is a frozen surrogate trained only on native PHOEBE 2.5.4 contact-envelope curves; native PHOEBE remains authoritative for computation and fitting.

Validated preview domain: inclination 65–90°, mass ratio 0.45–1.20, fill-out 0.10–0.50, T₂/T₁ 0.78–1.12. Fixed preview physics: P=1.8 d, M₁=1.2 M☉, T₁=6300 K, Johnson:V, circular synchronous orbit, irradiation off. Inputs outside this contract fail closed and should use `native_flux`.

The training/validation split is by physical system, never by phase sample. This notebook does not claim that a low-residual partial eclipse uniquely constrains mass ratio. The recovery example deliberately fixes q and uses a high-inclination system; broader inference needs RVs or other constraints.

In [ ]:
# Colab installs into its free CPU runtime; local Jupyter reuses an existing environment.
import importlib.util, subprocess, sys
if importlib.util.find_spec("phoebe") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "phoebe==2.5.4", "scipy", "matplotlib", "ipywidgets"])


In [ ]:
import json, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
try:
    import phoebe
except ImportError as exc:
    raise RuntimeError("Install PHOEBE 2.5.4 with the cell above before continuing") from exc
print("PHOEBE", phoebe.__version__)


In [ ]:
CONTACT_MODEL = json.loads(r'''{"schema":1,"name":"contact-rbf-fourier-v1","accepted":true,"source":"native PHOEBE 2.5.4 contact envelopes","keys":["inclination","mass_ratio","fillout_factor","temperature_ratio"],"bounds":{"inclination":[65.0,90.0],"mass_ratio":[0.45,1.2],"fillout_factor":[0.1,0.5],"temperature_ratio":[0.78,1.12]},"fixed":{"period":1.8,"eccentricity":0.0,"irradiation":false},"phases":256,"harmonics":48,"gamma":0.4,"ridge":1e-05,"nearest_scaled_distance_max":0.6631457345661578,"centers":[[-0.899953356012702,-0.8459318336099387,0.8206928372383118,0.1854499876499176],[0.35852911695837975,0.025425143539905326,-0.4543790966272354,-0.46711034141480934],[0.8997210413217545,-0.028846515342593304,0.37845530919730663,0.8461228217929604],[-0.3593727294355631,0.8493462800979614,-0.776011424139142,-0.50181083753705],[-0.14173205941915512,-0.48756530694663536,-0.1976417638361454,-0.9734426364302633],[0.6011244636029005,0.6361948400735855,0.5813367180526261,0.6323035228997476],[0.1425745952874422,-0.6376584377139809,-0.5014988165348768,-0.05788698978722062],[-0.6008913181722164,0.48903151601552963,0.14906148798763752,0.3368608392775052],[-0.6429636813700199,-0.15235680155456077,-0.7065773364156485,0.5226264726370573],[0.09842700697481632,0.9728453606367111,0.07288537360727787,-0.8638909272849556],[0.6432808097451925,-0.9723565485328436,-0.02381705865263939,0.44593804329633757],[-0.09774462878704071,0.15186917036771752,0.6262513883411889,-0.16708108596503735],[-0.40116298384964466,-0.5131714101880791,0.3296210225671534,-0.3261865917593241],[0.8558555617928505,0.3645561710000038,-0.9459356162697077,0.0446450375020504],[0.4004797674715519,-0.36211697570979606,0.9007656797766685,-0.6421328336000443],[-0.8561716210097075,0.5107276737689972,-0.2532087191939354,0.9865720663219693],[-0.7757564950734377,-0.2596065104007722,0.002554042264819145,-0.11960181780159496],[0.4813879653811455,0.611223952844739,-0.6520650703459978,0.27519345656037353],[0.7758896239101887,-0.6156169995665552,0.6967467516660695,-0.9120501503348347],[-0.4806435387581587,0.26400264538824536,-0.07847809791564953,0.6936218608170748],[-0.018495608121156693,-0.899085707962513,-0.8755976092070341,0.7844861950725324],[0.7230209950357676,0.21934158541262128,0.27511543221771717,-0.5633733160793779],[0.017752250656485558,-0.22569240629673015,-0.32369549944996845,0.24692067503929116],[-0.7231549620628357,0.9054300803691149,0.9554353021085262,-0.40568709932267644],[-0.5173021033406258,-0.742122031748295,-0.3998687379062176,-0.7038158327341081],[0.2256802748888731,0.39048836566507816,0.7503519691526892,0.9248111043125393],[0.5170078594237566,-0.3831665962934495,-0.8306781854480505,-0.26488315500319004],[-0.22633976861834526,0.734795244410634,0.448937321081758,0.1059921048581598],[-0.2745086904615164,-0.10016423463821422,0.5268177762627604,0.3843904659152029],[0.9840996377170086,0.779920743778348,-0.12730947881937027,-0.22867229394614685],[0.2751690149307251,-0.7745493575930597,0.20372157357633114,0.584065331146121],[-0.9838064704090357,0.09479355253279187,-0.5719875041395426,-0.8023741059005258],[-0.9418839029967785,-0.03366693854331981,-0.04175805300474167,-0.4353252016007897],[0.29962980933487415,0.8381783533841372,0.675697021186352,0.21722921542823315],[0.9420168530195951,-0.8415968641638757,-0.7198825273662806,-0.595998717471957],[-0.2988855615258217,0.0370842460542915,0.1172011848539114,0.751929439604283],[-0.20002121664583683,-0.6748282983899119,0.9148090071976187,0.7235617153346534],[0.5571184456348419,0.448443541303277,-0.2987395189702512,-0.882178941741586],[0.19927779957652092,-0.4499061852693559,0.34683913923800014,0.308005386963486],[-0.5572524722665548,0.6762954611331224,-0.9941510017961264,-0.08673653006553683],[-0.702718673273921,-0.9650918617844582,0.4230047632008791,-0.7724918927997353],[0.05588682368397713,0.1605985555797813,-0.7890753429383038,0.6140009611845019],[0.7024244889616966,-0.1601068824529649,0.8698819391429427,-0.1958027221262455],[-0.05654625780880451,0.9646070916205645,-0.4725689925253391,0.4171913322061296],[-0.44455745816230774,-0.325434446334839,-0.5499611962586641,0.07629636116325855],[0.7984201814979315,0.5518048871308565,0.16602498106658437,-0.29451019316911664],[0.4452179614454508,-0.5493647381663325,-0.24293319880962383,0.8922433443367481],[-0.7981268353760242,0.32299166359007336,0.5956117808818819,-0.7364370096474888],[-0.8200054317712784,-0.5718542393296959,-0.8439814690500498,0.36949014104902766],[0.4228539224714041,0.29848790913820245,0.4932550396770239,-0.025326400995254628],[0.8197732958942652,-0.30288000218570243,-0.41781163960695267,0.6619377247989175],[-0.4236973561346531,0.5762513279914856,0.79979569464922,-0.9437550101429226],[-0.0771519374102354,-0.21183606795966636,0.22093778289854527,-0.5346188675612212],[0.681335274130106,0.9070795327425003,-0.6202047970145941,0.8132613673806197],[0.07799453288316727,-0.9134274926036596,0.5408630110323429,-0.4970540963113307],[-0.6811020690947771,0.21818330138921715,-0.17283837124705315,0.15557494573295094],[-0.5744926575571299,-0.42936980538070213,0.7459339462220669,0.9536986537277705],[0.1825287789106369,0.7027523368597031,-0.09666990116238605,-0.674937592819333],[0.5748097263276577,-0.6954296138137581,0.04710540361702442,0.014766665175557137],[-0.18184646032750607,0.42204397171735764,-0.6651270817965269,-0.3561183288693428],[-0.31729375943541527,-0.7879145499318839,-0.36898502707481395,-0.13451716676354408],[0.9241017270833254,0.09265867620706536,0.9697122722864151,0.47855532728135564],[0.3166103642433882,-0.08728442899882805,-0.9240618962794542,-0.8341913428157571],[-0.9244179651141167,0.782546728849411,0.29207701794803165,0.5522574037313455],[-0.9149323254823685,-0.4597576875239612,-0.9589554481208324,-0.6694910638034341],[0.34208337031304836,0.6708688847720623,0.3250241018831732,0.9513994287699459],[0.914280453696847,-0.6654996369034054,-0.27936588786542416,-0.36135697178542614],[-0.3423699773848057,0.45438486710190773,0.8820548597723246,0.017285250127315743],[-0.15653775446116924,-0.8200268391519786,0.0859118178486824,0.48892092332243986],[0.5848557986319065,0.06204183772206284,-0.7019736841320992,-0.1475337985903027],[0.1568232849240303,-0.05471457727253448,0.6524170842021706,0.5421722028404472],[-0.5842030961066484,0.8127055428922176,-0.005097592249512672,-0.820905789732933],[-0.6592265609651804,-0.5399862471967938,0.5762438308447602,-0.0070912595838305],[0.08363090455532074,0.3288607485592363,-0.21018088050186634,0.3485959768295288],[0.659978624433279,-0.33520560152828704,0.1308312565088272,-0.9617097564041615],[-0.0834901537746191,0.5463365875184536,-0.5281518399715424,0.6831008102744813],[-0.41761451587080956,-0.18120387382805359,-0.44929484836757183,0.8422024417668579],[0.8408669997006655,0.9392069987952709,0.8332234341651203,-0.5604206249117849],[0.41747292689979076,-0.9436017069965601,-0.7577879317104816,0.12684175744652748],[-0.8416179940104485,0.18559834733605363,0.405101712793112,-0.47103297896683227],[-0.7922008447349072,-0.9347191713750362,-0.15529764257371448,0.603582093492151],[0.4664065260440111,0.1924666967242954,0.5060317013412716,-0.7594217509031297],[0.7928908597677946,-0.19002153351902973,-0.5829320810735226,0.42734509333968207],[-0.46608177572488785,0.9322814028710127,0.20095564797520638,-0.20915707387030102],[-0.03476656414568424,-0.2933068312704564,0.7783487793058161,-0.30002528242766857],[0.7082167565822601,0.5824369322508574,-0.37907413579523563,0.07866438478231474],[0.03444288298487663,-0.5819497816264632,0.45988862961530685,-0.7311450950801375],[-0.7089076098054647,0.29281753860414006,-0.827905647456646,0.8896712493151426],[-0.5321006756275892,-0.06555024161934864,0.25481006503105164,0.18834141269326232],[0.20941879227757454,0.8077903222292662,-0.9040817469358444,-0.4095769543200729],[0.5312646850943565,-0.8092560581862928,0.9521735291928055,0.7805940676480527],[-0.20964347012341022,0.06701431237161137,-0.33415948040783405,-0.6219510361552241],[-0.2894957959651947,-0.7054452858865263,-0.6317663230001926,-0.9003455229103564],[0.9676458146423101,0.4163314010947943,0.031031440943479316,0.7443870808929203],[0.28972130455076694,-0.4197472818195821,-0.07522484473884106,-0.06883505545556534],[-0.9668109007179737,0.7088652234524488,0.7072020936757326,0.28689581155776933],[-0.9893357325345278,-0.6470064185559752,0.17905142530798912,0.9196030050516126],[0.25364475697278976,0.4831319060176611,-0.5453575439751148,-0.7012667562812568],[0.9900256879627705,-0.4777636118233205,0.6217774767428639,0.11146912537515141],[-0.25332006625831127,0.6416326444596052,-0.224213732406497,-0.2671518363058566],[-0.24778376892209053,-0.007780019193887822,-0.8020951747894287,-0.23878798633813836],[0.5108188632875681,0.8747769128531218,0.41840784251689933,0.3976454753428702],[0.2474599089473486,-0.8674525134265423,-0.4987261611968279,-0.7920390535145997],[-0.511509895324707,0.00045586191117741315,0.8511711191385987,0.5710182078182693],[-0.7490086629986763,-0.352234933525324,-0.2936552707105876,0.25720822624862194],[0.008130056783556938,0.5161237549036741,0.9273396041244273,-0.09846928343176864],[0.7481728512793779,-0.5224695615470412,-0.975927509367466,0.7118264511227617],[-0.008354555815458298,0.3585843201726675,0.37348554283380553,-0.9329747986048462],[-0.49055210314691067,-0.9939570985734463,0.6706041339784861,-0.5920759532600639],[0.7509625665843487,0.12694208882749058,-0.05429716967046261,0.8105377964675433],[0.4907776713371277,-0.1313396580517293,0.09897095337510109,-0.3767155408859255],[-0.7501275930553675,0.9983487110584974,-0.7465355508029461,0.22114946879446484],[-0.8659929540008307,-0.24745708890259277,0.997709082439542,-0.8537066746503117],[0.3754034899175167,0.8802170269191265,-0.3481904249638319,0.509302664548158],[0.8653411418199539,-0.8777747247368097,0.3030283264815812,-0.1774694100022315],[-0.37569003738462925,0.24501645937561967,-0.9212893582880497,0.4590386766940351],[-0.12345006316900253,-0.6055708173662426,-0.12465806491672993,0.04990640841424443],[0.6335704308003187,0.2696846909821031,0.7251478713005786,-0.32878917828202237],[0.12373577244579792,-0.26919849403202545,-0.6760716587305069,0.9810264855623245],[-0.6329175494611263,0.60508057102561,0.04433947801589966,-0.6397953536361456],[-0.6222480274736881,-0.7532882038503886,-0.5999060496687889,-0.43847626447677635],[0.13623631931841373,0.12054066732525803,0.24941518157720566,0.15946718491613865],[0.6229999121278524,-0.12200926430523407,-0.169585118070245,-0.5307291243225336],[-0.13609574735164642,0.7547494135797024,0.5513183530420065,0.8718405179679398],[-0.3647536654025316,-0.39270922727882873,0.47294967249035835,0.6502354815602311],[0.8781085386872292,0.7285791449248791,-0.8724656067788601,-0.9945216607302427],[0.36461201682686806,-0.7319959793239834,0.7965339217334986,0.31872480921447277],[-0.8788595926016569,0.39612821117043495,-0.4282756205648185,-0.03703114017844189]],"weights":[[6.367211489721779,-0.25248941624609184,-0.9322938292652297,-0.03826475658875076,0.06860412285502424,0.04599689129611814,-0.18665544515100485,-0.022319490888091285,-0.101409435487154,-0.002039990948345612,0.010764247388658341,-0.05964424804191084,0.017880484604845433,0.021021305412263727,0.06843085521095295,0.05860836864410426,0.0750415017185668,0.0674084895127807,-0.0014786479215812044,0.06169701257676419,0.003009349476033639,-0.03244730349130567,0.017805192091211532,0.04749841251036591,-0.025607793923449043,-0.0181007149600959,0.03491183820878189,0.04051187521099671,-0.003811164238467532,-0.02848300375980511,0.011999694886141315,0.0412542472123458,-0.04444666738749159,0.015281268439814474,0.05950945174641585,-0.005379179921570899,-0.005242907583620081,0.05680223040892971,0.001219080827850518,0.028403371739190626,0.05294020228530641,-0.017373919506603885,-0.02475383359169131,0.010255991770688807,0.028515937389814845,0.007673713058841496,0.001633358009700499,-0.020227389692082404,0.03179499629837351],[20.627600288765635,-1.2799990015573062,-6.648964182830538,0.2601878954475184,-0.35559362480045514,-0.48242638539449395,-0.416126565702167,-0.07474232520013312,-0.8811052199270556,-0.28591077446474955,-0.611747282430865,0.3434823947241413,-0.3796472051116836,-0.3596777181431321,-0.4677832909397968,0.014307560105144062,-0.6203903199486499,-0.25655708415789164,-0.1733758833415698,0.43029726575515614,-0.25382164308693916,-0.02358901274883265,0.14382372135629662,-0.012126806436431612,0.08501473970656213,-0.0488993419270124,0.031994797154193635,0.03695556765424058,-0.135798726462481,-0.36240435894998646,-0.2578865835911536,-0.000203345497679476,-0.46065416987832003,-0.20977769608066327,0.07077503627442644,-0.2635112437106903,0.1842306745394841,0.28038302078348987,0.1634262238310403,-0.22273341297946414,0.2976408574824553,-0.025605353816069862,0.02710906652194381,-0.22548913942670692,0.028326667781062752,-0.10783844260378801,0.02830645985536547,0.02462066839277104,0.16670886186541303],[1.8239486572233552,0.13111842597349788,-0.6462105093689126,0.023029257362226226,-0.13685764136165648,-0.04114934386780835,-0.19472617072767842,-0.007506465969892775,-0.10700115833298689,-0.051403129017159224,-0.11621457240561434,0.008679594010030548,-0.10694498424872326,-0.054695373478470546,-0.15276332873274767,-0.037862489194769255,-0.10456243949369175,-0.09045730555041268,-0.11418980457650504,-0.050766020552315924,-0.08563147589874007,-0.008415681318860743,-0.08104243561591194,-0.058469694378008356,-0.10353404099970574,-0.028902784628264535,-0.0814429173922547,-0.04755187232359095,-0.0899166165615602,-0.04091240563986672,-0.08983209956831212,-0.08036645369169484,-0.044440848196276014,-0.08310075874394039,-0.039490507328519796,-0.06081346922822087,-0.010094029435363482,-0.0658464291364271,-0.014736395355062385,-0.06610873456135621,-0.028277784963492206,-0.045763487557216556,0.016816198088362405,-0.0647953002500585,-0.01976710950126573,-0.03991812623613417,-0.015445301294051141,-0.030530424552565924,0.02846706465795507],[1.1167855193682328,-0.18468112684779517,-0.8877191178505177,-0.18497231079262005,-0.020015421566706524,-0.07373575420443118,0.04997952489078601,-0.036732162472996904,-0.05324863487208737,-0.03516255978970549,0.0026090734469741295,0.0709218584344226,0.007936369543690903,-0.05939630113287498,0.007829102426679387,-0.03584382418780042,0.006793848130884159,-0.02067942237789512,0.010990632823799267,-0.0019419318575169446,0.0539437657176485,-0.06028998201693432,0.059822296839612314,0.0342904476014696,0.09811892969392699,-0.025090330671965137,0.045099202078877104,0.04755098907747055,0.01779447057360725,0.00809485688564643,0.09488399047404522,-0.016811229656274084,-0.048713093570798074,-0.042589467994956526,0.05569597138509252,-0.01417698192323615,0.058588184130215284,0.013035086929647608,0.06821171263395844,0.0005805862519878539,0.045279788569592486,0.008148100780102441,0.033391131127515114,-0.001974489324114599,0.06846214187320858,-0.017336180142442516,0.029943567587222907,0.0029966481473055443,0.023787576646317418],[4.870781549067058,-0.6482399603941088,-0.3999947520877771,0.010176424278380997,0.2427175796972464,0.276192263138716,0.26119603296765004,0.08631004894604605,0.1527252916639665,0.08656008519105879,0.24129887659146051,0.060347503964833585,0.12321082171757067,0.11077628337426042,0.20383028569974476,0.06507046308142234,0.14182255577280897,0.08581695870250892,0.148315375634509,0.14267698805679488,0.06152515153163968,0.06691670693513294,0.04241660507024433,0.07738777675056527,0.0053748670780675535,0.03642329345869856,0.0542742811594845,0.03379506161209715,0.12128374145581901,-0.04372028377621608,0.03897408779416888,0.05193241920805941,0.06650333307901359,0.023992418321481565,0.05140903322916638,0.015180259704274994,0.008010945962859256,0.06376075253802808,0.017414041738188304,0.014077273112400996,0.021672799600063793,-0.024264298165612812,-0.015526650691043004,0.06765356583287667,0.011699270099829445,-0.009983133220252372,0.02605980982246728,0.0009258248632277007,0.01295276041524732],[7.552103810618926,-0.5869838073532593,-0.9675914161912096,-0.4532172013695534,-0.6736496303676079,-0.20581331526171098,-0.5831576144175822,-0.1896305888878929,-0.11417836525017164,-0.016090526553448822,0.05489774066530962,-0.018075273422261676,-0.05873342766309522,-0.09507997656057736,-0.04582497135954141,-0.02692238866211451,-0.04464154203746019,-0.16560943494126756,0.19330196720574963,-0.05817457494787064,-0.006288803449484515,-0.015214253384282261,0.12697251030725454,-0.012754138014583236,0.01879256905236603,0.026913652215132842,-0.004167466122837957,-0.08050082041408257,0.07146997128795271,0.015241610916808231,0.08208695026382509,0.048132861367347325,0.0781949395564858,0.030578256762759812,0.11622191976047727,0.024617937335107736,0.07473337232759364,-0.07495621758319079,0.05802117540026939,-0.003649886065213247,-0.009779839655401648,-0.028437334622078425,0.027184256056667636,-0.015380559751223623,0.06125942504347381,-0.032231070117756166,0.06167217795365134,-0.01073788490206742,0.004316891975723715],[-8.374095507236403,3.5725120036761524,3.9433992738573433,-0.016277927149237825,0.4281904450139908,0.28436132112572043,0.314218101251896,0.6724896602263253,0.4642141514144794,0.6539370075365053,0.9354291461310854,0.4468310513811799,0.6410922065792819,0.4100212575914141,0.6517200894182495,0.016250250125990708,0.31581985492616566,0.3978654019577666,0.6929223804496757,0.10714468554318465,-0.03201964124332919,-0.054183895808550264,0.02689298784741438,0.26393677845090346,-0.03274349651788606,-0.006501044402473912,0.020259529966828137,0.036476658447520544,0.22691759105035766,0.01696182373667791,0.303124599980348,0.1950522158024221,0.045913403178769435,0.37410785084639064,0.06391396029048435,0.09692324180584425,-0.10586736015853072,-0.07320787662428017,-0.05585516088031074,0.1120436013196509,-0.29737021300748334,-0.041165489375918594,-0.23516514694693394,0.23441300332353995,0.12339044172823722,0.028107298856652613,0.07303456209714211,-0.11037381512621966,-0.005144655459230887],[-13.927867960644331,-0.2576220171962831,-2.6603894873542315,-0.5833692134974133,2.1131639288935404,-0.4960148588549427,0.749411191725813,-0.7763170773400703,0.3365717832011069,-0.018633443964239646,0.05818294619338779,-0.22861096212570306,-0.0053766510807444975,-0.016072313695418942,0.027514310877303893,-0.8496984723708519,0.06579720667533395,-0.2162171960208076,0.016203400087310377,-0.5054827661687403,0.06930280428013721,0.39195823481981396,-0.14828151574725046,-0.19405766759460719,0.07711497435718229,0.17898818355072965,0.186731236917631,-0.001939044729143964,0.0708651035245897,-0.08559994240503822,0.05836319214680734,-0.07044946914110178,0.13535796867622318,-0.19276743931828047,-0.100580655445447,0.13640758766264244,-0.0827429448716072,-0.18545289299158113,0.036245452375609485,-0.03303047280072707,-0.09062392216431685,-0.025556842285578755,-0.015912020231661748,-0.03552203833770394,-0.31651874135601915,0.02889406887520187,-0.05586707377989365,-0.03280104930826393,-0.07192129708346766],[-7.701106055365644,0.5935463613547955,1.4391111291061782,-0.28307987726425554,1.0462243875929822,0.19229057427852042,1.0731680478301433,-0.08597508032824627,0.3879056586487191,-0.16012575077174773,0.36060135239165064,-0.09606573502474601,-0.17062074373875005,-0.08337818922410509,0.3622437965348574,-0.12783408671518576,-0.06273625812586699,-0.054096892761636886,0.43443933635112014,0.0973403848078092,-0.1304306122558736,0.1985457231027504,-0.03393051112667187,0.009177759238080953,-0.06595779605012991,0.004598954082691957,0.17953889105177698,-0.09086237791669202,0.026268161458828296,-0.17681467398903858,0.042114972389228185,0.10841164038754005,0.15489044746050004,-0.08395100948102657,0.10733320558174146,0.11944546429789847,-0.06734338443200936,0.12365363369629744,-0.1649846780921872,-0.13935298362863155,-0.0778632307219224,-0.04100418153175317,0.02358081143514125,0.13449062790893002,-0.08338913861738204,-0.025405725696236035,0.006133901576958355,0.08728504802254244,0.07613252561637161],[-2.3095142296070414,-1.3505065855477687,0.22646416669437647,0.6641861699167455,0.12329052182062156,-0.04740793855505151,0.011042168977964322,0.06328315518264904,-0.033400862582565236,-0.2088016381025916,-0.11975751128518855,0.16288076313172783,0.03446403401070479,0.06699274479228537,-0.043627577833127165,0.024920556883979098,-0.047772563845266076,-0.04067145486335917,0.024232182522739943,-0.03411196769170659,0.020065315208736147,-0.060003232677795185,0.04743978864276543,0.009152567348745297,0.03369847457630129,0.026444232546807486,-0.003050517660784623,-0.0352533088154372,-0.06353690004955145,-0.043931986524667316,0.009802192813905352,-0.04462458950492028,0.03511066015179819,-0.0634685106905988,-0.025136533273076425,-0.03797677249172258,-0.0014449452946928816,0.018767945260269425,-0.05301891280990582,0.007603371375452636,-0.0329750377359561,-0.05287607102807214,0.018086391107759416,-0.03824309786580876,-0.0004600639380453376,-0.008858269987573783,-0.006019184761445989,-0.017103956454312442,0.007726023421564423],[4.918733351767164,-0.04508840360696456,-0.20802493722152426,0.026515344077320983,1.1475458401808656,-0.05472852530925651,0.7242823229382539,0.16681745604663387,0.8468735565853226,-0.11375273771761925,0.7523962686178115,0.10089971470040113,0.6687549331198606,0.09172805704196664,0.4602792909986229,0.0799193088023256,0.4028763235211926,0.03897843783015425,0.33027672557630694,0.017370810957260046,0.24275867807221377,0.21641665404487961,0.1338134745467191,0.02365942705374007,0.004340156439049927,0.2160805381772242,0.02845245927528968,0.034869966308718706,-0.014507582966513567,0.05546063640637769,-0.014307878447635234,0.1544730228392028,-0.025435550423770464,0.13993373320578434,-0.07485835015712394,0.06343642554599677,-0.04911485581972895,-0.025972119026547162,-0.004807948165295674,0.02520717532609706,-0.10009299008114576,-0.02336637968333863,-0.10026535923139762,0.01064225823933347,-0.158965576227174,0.03133216520925265,-0.11709465457443112,-0.020089303862196775,-0.07162443539581242],[22.338072992606136,-0.46464992693291907,-7.057299083263203,-0.9457573551325471,-0.4947754360217163,-0.01756002786244377,-1.5186230150486504,0.04675174348115579,-0.23802986735769713,0.4343833909616704,0.3082664255389839,0.03733622758967721,-0.7735118383568831,-0.1809966341125447,-0.5954981171408108,-0.6462441543259287,-0.5695704101346518,-0.2871934040475041,-0.5439431904380503,-0.5365639921445884,-0.5881787084399529,0.546281889255863,-0.24844218055660944,-0.3428789525561917,-0.41728657848490025,0.35852010933299805,-0.3641078783930133,-0.012088158590749624,-0.33713601899422135,0.16012574034350535,-0.342070237244665,0.1712040463949282,-0.23992222865024238,-0.09248614832359957,-0.23613426382586747,0.17691851673383727,-0.40034060702820207,-0.49439332443128,-0.2696633999349388,-0.14643559094378186,-0.34435794367770833,-0.2131566215865963,-0.3697882502147612,-0.20171800511007182,-0.49561078799684033,-0.16819264238915993,-0.02092627235425789,-0.18494739515825603,-0.19983737741713486],[25.537527278995654,-2.8767616491091488,-9.814268357636177,-0.5639113309445403,0.543432544600118,0.29200259633670755,-1.1476143764903117,-0.19915824816417296,-0.16427684642633075,0.46333462630989464,0.3492048688646729,-0.1857135760097739,-0.2227898709494963,0.1711204141408479,0.05847474113031254,0.004891588725561642,0.20598179974855305,0.5853118109862383,-0.42096757163844134,-0.025884969225237302,-0.1259480526709708,0.14794987857100272,0.006764545133643148,0.17169028231498276,-0.29181142814592154,-0.02405464107594467,0.054317991270892346,0.34762293632370067,0.00632507966335741,-0.07656655271288686,-0.1018452550004832,0.18630224017319102,-0.43927631120549454,0.0706635122271759,0.163699501653217,0.01752921142629305,-0.24554680249673766,0.15114404369696505,-0.14442351638422282,0.07146734781331383,0.1451938483891734,-0.23935499378615896,-0.2286733858518628,0.033957058762034024,-0.09768266620563057,0.07268087966254662,-0.03500641442564477,-0.17403578781024218,0.09629412103956088],[4.866807387847618,-0.5438869262114792,-1.4605983591812817,0.477275653803753,0.16610808211364847,0.026348049437901668,-0.30575687729938456,0.022377993376690653,-0.0956828142005195,-0.11406251062865994,-0.11862653854549636,0.007496584808096677,-0.001739139185987399,0.03754034660303957,-0.13458806352135846,-0.03658959260122868,-0.09949587499479245,-0.04645851281447273,-0.09968782039802272,-0.018501294035896492,-0.04328476131284847,-0.007352993554659344,-0.04287827061070935,-0.0328582883158178,-0.06348407832721459,0.05076937638882659,-0.07284069248722606,-0.009002737355529247,-0.03595851387568644,-0.03611930567082482,-0.06971217087206073,-0.023129113526075783,-0.04148431898744325,-0.019734092343740767,-0.061521760341864656,-0.0510182763212512,-0.023733020476522017,-0.01985697507890379,-0.0033205792053518247,-0.020582653641609005,-0.026586937898184265,-0.013199586562491029,-0.03330304588030183,-0.023667733834341256,-0.05917341713150784,-0.004177009923400896,-0.03138325685132306,-0.027529849393541726,-0.012086143350166742],[-1.9469019276230555,0.618523105095526,-0.604977198113441,-0.19998161192844752,0.32783932562097295,-0.050434214420987285,0.4456963492042818,0.1802547555893014,0.1830183947033782,0.18296608933228486,0.2154649538962856,0.1984409812090267,0.1540633525677689,-0.00022784952278233545,-0.1445572680194296,-0.012028370043515785,0.03398008590527659,-0.03478138194376004,-0.12471260244063112,0.024745380246784174,0.04927370850486098,0.17356698634803178,0.0032666320487513816,-0.008366090649938988,-0.056496763490482355,-0.03940908564557991,-0.07096968611922906,0.06960239624169691,-0.09608241954768558,0.015197705411364478,-0.1283304685618817,0.06360961697074491,-0.1401841618175853,0.04365448659088223,-0.14198934451991055,-0.06479165695145503,0.0384729193644542,-0.03206077308062619,0.08749546331677781,-0.06656541521764295,0.028100362280287026,-0.007266252589041632,-0.044133577405574796,-0.06947117072080948,-0.06076286272962372,-0.00023563417214907814,-0.037486106613488446,-0.0018205009129800096,-0.04228873756504777],[2.5812564512358036,0.3081992283055282,-1.0431945412918822,-0.21672640926848497,0.45249859541842147,-0.056000758326602826,0.10154493712134466,-0.09302212732044537,0.031906985130632434,-0.09470061289496133,0.019479322977934515,0.025838175878484356,0.024620103335709742,-0.04382774383780189,0.023638636299001793,-0.09592481114648072,0.04218525652418209,-0.04380076468984967,0.05389353591462085,-0.03572570354705245,0.014461317889259454,0.03949521565878944,0.0050292657649222165,0.005604084487146566,0.025996833403948802,0.014221640465141537,0.0648188568126401,0.014711129864874363,0.010884273857594861,-0.02416913901390436,0.05186450691163696,0.010810439860577664,0.0035433543242795284,-0.02582249405300543,0.021785503955063935,0.00019765752565179093,0.033591525359044604,-0.018275672813450355,0.050677510601415204,-0.011054884413468438,-0.007615229950845821,-0.025345720632236377,0.01057193289122688,0.003267513909946946,0.0004203996035129154,-0.007661571679034697,0.01749584400367275,0.0007730877409702465,0.008975774429229031],[0.40473975879191154,2.6241967953710366,6.414168679680316,1.3778170218937047,-0.377791952694026,-0.22629865854648326,-0.10780429684045517,0.5444110839831307,-0.256811523107277,-0.7809296048114382,-0.1353791346548383,0.037647449823478125,0.9874837415533106,0.0025305210546422686,0.68407304124457,0.22917036806213698,0.4267457740567961,-0.6342257129098023,0.7637181201681444,0.6274317442970021,0.1839923579382566,-0.49838295910021085,-0.07904952534249976,-0.11232304375954291,-0.22027434861930792,0.17132564418797955,0.17692781692332674,-0.2854279433128781,-0.2764737790599001,-0.22255273322940886,0.21611127276723388,-0.05216544369528884,0.397772697332563,0.018587002628620394,0.028669303303952377,-0.09241231301651702,0.24154138315771123,0.19048335628572274,0.11926055515153346,0.052067844036010176,-0.16835233233902672,0.2939926725970687,-0.03570885691476886,0.02205168616281248,0.2713616988047465,-0.05720188366614077,-0.04258873712888789,0.13329873336604495,0.1061338069271189],[-12.640585243167806,-2.155422491016938,2.136813765249424,-0.31369539050789524,1.0909104900363762,-0.03507328008437743,0.9185396914870346,-0.058027627502610334,0.44861471943226106,-0.17834986657018834,0.12024936165110624,-0.14803067642276152,0.044608147522614286,-0.02509008002976862,0.16350399443356992,-0.037486870032543806,0.10920746602472096,0.18223734100738673,-0.0037640846216763276,-0.15153131929462452,0.16084178213549097,-0.12562323840975156,-0.02555963418198338,0.029492396608163482,0.04085299049382216,-0.041008860272463594,0.04974131530228676,0.012026162736961806,0.026801422122841367,0.0008812100935825392,0.1129474652494923,-0.15941450998084122,0.024478202427508935,-0.10947054448596902,0.060342888561305784,0.053393739337725704,-0.021819837022637546,0.1096035625069024,-0.11584541307524879,0.031882340067198284,-0.0062615648295590385,0.046547896944167846,-0.002096629330537301,0.10618691208818846,0.07892929246660935,-0.0007371009255504642,-0.05213624290939764,0.01543563060288105,0.03510684072450537],[1.2600223897887035,-0.4537231850329063,0.41837794451093047,-0.14345054862456177,-0.104289048089317,-0.03467250207060211,-0.10307477823390894,-0.05154896453384841,-0.05248630663604357,-0.058084007086423306,-0.07165707382530452,-0.05844156508628782,-0.00031716894731969485,0.01405716037943136,0.08331099565509072,0.019216304797391493,0.029180397061119745,0.05282279566685953,0.07900698102169167,0.025904957987160148,0.04101474599281464,-0.05170014785814774,0.043189723376028104,0.04465342785936466,0.08979483631708322,0.015585333395902085,0.07308634769065983,0.006456000957722097,0.09805999601233678,0.002989765703355904,0.08751081682726165,2.3039083601619013e-05,0.08249916526748566,0.010735146826367855,0.07628059554876494,0.01986447800098735,0.045202278829815354,0.038696022371399776,0.025448303183071924,0.038122722064568994,0.03486286648310901,0.01368468491693227,0.044618488215152165,0.05480623595432835,0.04583617481438719,0.01727900961650944,0.02248570998468264,0.014690482714273116,0.02803564089661073],[-15.431070627136167,-2.725021542415977,4.8463129357076316,0.8866388098382926,-2.626453771149612,0.4194438304452807,-1.5014387661615782,0.4636320935055901,-0.9427846029164199,0.33553409191767625,-0.8182088036124967,-0.39951995859736394,-0.3142877876699703,0.16914289868217997,-0.10422562105862727,0.5502505331437192,-0.5371667315458688,0.4880016216397379,-0.2461261561524025,0.25113420639362005,-0.3165865357794861,-0.7205174050971146,-0.1952194665428211,-0.05360660545447078,-0.10777321900966534,-0.2707075584505823,-0.4742876479459767,-0.052253917422147425,-0.16119200277795706,0.2862850073136864,-0.19709446078445517,0.07532145502000784,-0.06736843902763953,0.16214542593485115,-0.0669329412296663,0.04332599647101865,-0.1964672189192732,0.12319958078172684,-0.38692196125348755,0.03938143721483253,0.014274036221575836,0.23970690217264412,-0.1508643910873165,0.07558422848049828,0.12581901484882602,-0.08034174159682343,-0.03239270692540606,-0.07568257708154458,0.021259997329261864],[0.9939054450690343,0.0018141904506411158,-0.38990546476691834,0.07995856156476683,-0.1265399363901688,0.01168357968261977,-0.023366495801852363,-0.0028670402682614616,-0.04641313156733177,-0.005577182830407832,-0.032796784066712735,0.008581236141696959,-0.05342991206243631,-0.014932873753082655,-0.038411195749711594,0.01826258052300076,-0.01575036810173524,-0.0099760929056275,-0.01182532636204062,0.022046426374748657,-0.008225968419405837,0.01110878535648218,0.005519660511782853,-0.0032505957996961476,-0.0036188914197123716,-0.000990382878416497,0.01987518698621628,0.002112091071501125,-0.0032471189456872553,-0.02140684499273864,-0.011755576405093511,-0.003268381359278887,-0.006027514681959411,-0.0069382929604040145,0.021093923054398646,-0.003133597064111527,0.009490466997367749,0.011923666476334164,0.008597675465967689,-0.0026578622098680183,0.021061020562890272,-0.013288281038122185,0.012487552489146993,-0.01214611429210265,0.007961206141197382,-0.0011236173230844244,0.01272474816243959,0.0013450613572887611,0.011119912832768076],[-2.136389170327838,3.7059239913106468,-0.21386545311688696,-0.42506553771719374,0.747500212889226,0.3779207305006067,1.5689424029779266,0.5927483173923143,0.10709775274183586,0.17812812213880697,0.2856856243186553,0.4338668403236282,0.19391039164021617,-0.421108846098549,-0.15631918407453158,0.1424502381505658,0.3306486467248291,-0.38648669540452674,-0.14064866703045523,0.32839196661012654,0.3831229944622802,0.42955884024331653,0.049601440127014514,-0.10326291513970634,-0.04160130660089869,-0.03592096256638401,0.19223822492558673,0.13967874055507382,-0.1668999771117432,-0.27653552283928184,-0.28789310037601534,0.13965522498855212,-0.2784326596350499,-0.0728931381632765,-0.1524706201272844,-0.21808864198581496,0.26484114984118406,0.09461897825693118,0.38884137555562587,-0.16195933787904318,0.3666315196861098,0.03126517377241762,0.15789788684306524,-0.26485075415912607,-0.006003949761408031,-0.0013438083885328486,0.05003820981444461,0.08201817683414986,0.0776084124085693],[-2.313902390642897,-6.47113487189782,-3.0551128605483937,0.4984907976185141,0.6860202026207605,-0.3023039490944952,-0.7745293119579282,-0.3344648782817762,-0.5444629155608939,-0.9766050666404483,-0.9120722601920197,-0.41616425593231576,-0.6535491197796546,0.08389433481425616,-0.12437247786935145,-0.2373345212244843,-0.08726360508504306,-0.09280981515252817,-0.1831415820082208,-0.23838040729489157,0.22935947072461932,-0.34505923955082585,-0.0040468283210027345,-0.10443326833875848,0.2843917094341061,-0.10539365915714116,0.5330100084300149,-0.07493170000859996,0.1767178172547438,-0.09733783988248662,0.32493880898242644,-0.26182154051906625,0.186163818952114,-0.24374204917131373,0.4493841527165721,-0.02491667997342648,0.2961142224693214,0.15728282634982388,0.05583075403364794,0.0438839314263943,0.2866633272905422,-0.14600551760512845,0.2282595165119321,0.17557375175520845,0.04092346463877422,0.09892832594039651,-0.03236200917442158,-0.024807181710591143,0.2178955489739661],[3.1575670152094784,-0.13688500038861523,-0.456521435372157,0.06298336735888933,0.00040289924127877774,0.02026167444576,-0.053183029544363186,0.027220903352670082,0.006816948963412734,0.03120754531790682,0.035870904219282775,0.02993924508927213,-0.005449117376446068,0.010135358362909035,-0.01408746946154708,0.005046649884438397,0.021323657317693666,-0.015715913728025645,-0.009531106150220863,-0.013791724031305515,0.01625977550543008,0.03820499148619734,0.022915432417024467,0.0014915362782198557,-0.011771169615714165,0.032566027165940155,-0.003810162208812249,-0.0019722184723345425,-0.026716518561639046,0.002197262593376504,-0.010431323032953535,0.007769923782777248,-0.007400142957600664,-0.01406411318069998,-0.010171754238753819,-0.008162442474916097,-0.009268783251321775,0.0041613944447553034,-0.006614326250225502,0.019148921487661674,0.00659778970275286,-0.017454658671891436,0.0011528493003569744,-0.020214963451576876,-0.013944924465613998,0.006234561673342987,-0.008332972214888082,-0.005880791907998635,-0.0003113237421059823],[-8.987856334952193,0.8455794203831961,2.2649107914892954,0.14360555350218024,0.12127711556308207,-0.1359645855122828,-0.4834022604808827,0.3197116814132515,0.057970073803993376,-0.06390037844868286,-0.03163058831985058,0.06404464267193716,0.1225381613749404,0.05383957457098724,-0.11501934495648516,0.07092731409478807,-0.1554111231993799,0.004187719922332459,-0.11018951123160674,-0.07465595889004696,-0.07007858259651191,-0.05327176998587636,0.09160706249910096,0.0009491878684682334,-0.0633044092130425,0.04701978272344523,-0.17767184355033117,-0.0465089398727383,-0.1548880283126602,0.08829791495694667,-0.03916471528623164,-0.047685502202031674,-0.11708621719799518,0.05106572305718778,-0.16935121547162255,-0.0325605174858131,-0.031121374430986146,-0.12292824653861088,-0.05419984196473361,0.012353900256108002,-0.10313824345913562,0.024933702494792046,-0.058786578587340044,-0.06753289750834816,-0.04338448260438479,0.0034893098240196237,-0.06367391278949236,-0.015426898385600227,-0.05903125202512678],[-1.1467404347761114,0.6768580517631315,-0.6998611413856891,0.5240668271330279,0.08341871428342953,0.06938027394583285,0.013269652761849608,0.17242924036830481,-0.08204118971449748,-0.1033950526414469,-0.10848875259542319,-0.03592213269774934,0.08045750296279382,0.10075350061492058,0.042875558333199504,0.06946438176457921,0.04818573114532921,0.03679222026345814,0.020191199395096344,0.07577493488156317,0.03350764834218838,-0.043442463753538114,-0.0027858419176762677,0.004746514565715311,-0.010922079756949827,0.02758455722236765,0.04713908005032019,0.02991717896830892,-0.053083906716329864,0.0072142127934416685,-0.006899435079194519,0.009304183126396358,-0.03076789807576447,0.0477498143522817,-0.02338119277518682,0.02199820119260552,0.011714797003868415,0.07965869719841287,0.019156864359148457,0.03171620327861969,0.00920898783070747,0.03652934271495958,-0.0020825777386995884,0.014067226853770735,0.007792383995254045,0.04274378208343691,-0.03313559623991713,0.011990294160552336,0.01111614432839541],[-0.3470461896171024,-1.4247104546172813,-1.9055068042464443,0.20512695982071352,-0.09846804328620863,-0.33626971591747645,-0.5314323743774267,-0.3944544327629619,-0.3568079025365364,-0.46934235822147347,-0.6272802051152708,-0.29999996531330236,-0.35969689745801714,-0.19055310785259716,-0.3326292095605236,-0.23374221904821926,-0.31579738176961325,-0.18870921957492875,-0.3045850547122399,-0.10589916268875542,-0.16888505850796945,-0.03270157093866215,-0.11334350204613684,-0.17683549724703765,-0.13005324133466223,0.0013173582733725393,-0.06900539516495387,-0.08563368156417978,-0.07421607198641365,-0.10975743918362683,-0.13100643853502106,-0.10737851651326259,0.0033368239826074403,-0.12816402464183438,-0.043859643490439935,-0.04044018487806339,-0.009396516502792643,-0.06785544240520323,-0.03853123888352022,-0.12945157997081058,0.034309531462632346,-0.007899904555210717,0.01018261736386648,-0.042117666465973785,-0.07244692462374336,-0.030502052638803935,-0.009520450291681757,-0.0067046338661042515,-0.004618253753920688],[3.4424286482591473,-0.5394798247698737,2.588584857370032,0.2415325114432824,-0.14556163615157822,0.15735709420887167,-0.10483385387899673,0.545966717330093,0.030057012068053952,-0.22034297482710438,0.06850482685475544,0.3035369009654814,0.3459478654566721,0.11734105214192572,0.3021138220394465,0.6329857796177211,0.3284682885030609,0.05360971779155561,0.39381478976773143,0.38682140271675886,0.17650752378683657,-0.125105221064635,0.3006873764540266,0.12905619877352958,0.14464961440071847,0.10629473807726314,0.11529778575915364,0.008696940759498527,-0.02003247348820656,0.07712411829417065,0.2083985189566573,0.15756100854337088,0.005279025757204509,0.18281362624920805,0.1460260703892582,-0.0813730829504773,0.18080612724901235,0.1413737073829296,0.14614864856740764,0.1187044838914424,0.09260944822175035,-0.06078829264994956,0.08948011095196917,0.012790151249064003,0.21012321569166595,-0.015648704761668852,0.07081316056282025,0.04995881623449951,0.07633052696044369],[7.815090934793036,1.575416434540441,-4.918880745258224,2.9784430278162386,-1.0989318578609557,-0.9691637376174345,-1.654235548339281,0.9171442527442911,-1.6606091841782287,-1.6054423684370929,-2.0981285671658427,0.5554229381539236,0.7152966625405047,0.0410217128172832,-0.8023810759452564,1.075700906842872,0.8566644997623002,-0.6466998650118548,-0.3309904960230937,1.1683304878229899,0.470827615045466,-0.7060521488186826,0.3022688969108392,-0.017335249139222957,-0.06789921912723866,-0.4539004774694767,0.5842932107663117,0.2132671673622845,-0.6274969035641574,0.17608942996817262,-0.20792504828772615,-0.44854082202608697,-0.320362567089386,-0.2103122666483199,0.10753816825816237,-0.5518564938628082,1.1278706811070267,0.547355443841253,1.1017273658500613,0.20210002688082157,0.449133577214147,0.07624998859084388,0.3746543788675976,-0.28991456175084473,0.6002040804808335,0.30007098540327437,-0.15034090119454227,0.23163311354320448,0.20223916610963094],[0.3383801748339101,1.1784133548209235,1.5892935067553748,-0.1573593059515549,0.4857141318942231,0.37827128507812185,0.7984324870007746,0.33382014398479926,0.45482280360351934,0.24017792136978203,0.4551176468074777,0.0806856951581557,0.3142322191264381,0.03921473082999115,0.2675483892605808,0.2000393505740298,0.31918052205296377,0.19034844857887256,0.15839749215404253,0.1431001786062732,0.23833064628750594,0.06556685430703336,0.10518993203694255,0.15611916987091123,0.09884924994307515,-0.0010373639570754128,0.1252124460058276,0.11223472617969515,0.07608778835426,-0.007454988546564517,0.059424486853335234,0.08660229462002113,-0.011477888737652054,0.061806676651134554,0.0531738694588442,0.02372821366384801,0.08059828377845883,0.1336681846928367,0.08427618076134548,0.08584412539019365,0.12922700691168237,0.059204328652228584,0.03595798910881731,0.0407965637506731,0.1032525525788683,0.034825733657589446,0.01944224931762657,0.04579786900317745,0.053824442561642045],[1.9768348706396186,-1.5675994783422036,-1.6361612936842054,-0.7963809304029316,-1.1117029114068857,0.46224458532024204,-0.9132647217382038,-0.24792063982626422,-1.055738095561585,0.18586284194213548,-0.8763053626223924,-0.10968233142756176,-0.9349477802797403,-0.14597426587151469,-0.5286436085794879,0.1662776022044684,-0.3092874949819611,0.17996350745488568,-0.4173900867374585,0.11062729865691268,-0.17771649297608047,-0.3295662064977739,-0.0656535471683838,0.10387429982094808,0.15751462161438953,-0.4359656441262815,0.1693008230224458,0.04334212476327232,0.07796001142272647,0.02690885783623471,0.11650161444562049,-0.17283066094833746,0.08323240659625668,-0.19003223349542714,0.32939549134585083,0.03580814639508053,0.05146121597446913,0.12009609356995198,-0.01084689155017548,0.04143488417922891,0.25692201258254155,0.03277551818295667,0.1595604516698292,0.06553806832118568,0.29449219029820695,-0.013215667011181749,0.24641898046075184,0.01742652190745352,0.1654754837675348],[1.5675257000400868,-0.0054313908223416986,0.33670515570589193,0.10287635251169489,-0.01501846107514378,-0.05009821052035357,0.14681031920590662,-0.022056246788032835,0.02665732594216145,-0.053533175444065224,0.01678753498195962,0.06893832184922959,0.06998701036893543,-0.06153437875306895,0.004972837217323068,0.061041877968172224,0.043094962970196396,-0.08020127575332225,0.03738505977225231,0.06467961951639313,0.052140136671043225,-0.004723854844040792,-0.006900826801274661,-0.012727575335599017,-0.01420663098982557,-0.004657105414835296,0.01371954915400774,-0.03185212182234105,-0.027875141182481068,-0.053767570861684544,-0.034395027932948186,-0.03360508351913017,0.022065540175686578,-0.0216956444797917,0.010711569750894715,-0.04031153375762639,0.04906242562495975,0.05178259607490701,0.02945929713853877,-0.0007054066913982871,0.01459658961950722,0.016132461987495086,0.024755626018283117,-0.024771434078830343,0.03056063035482016,-0.004782012869738872,-0.026470340070385536,0.024797859634601474,0.018696883515498027],[0.7867913646543423,-0.9008249906842741,-3.0949459482961363,-0.7037016272493101,0.30071242037621687,0.1865250709246197,-0.06697835536935963,-0.17692009415008741,0.06855871003736179,0.33135507495967015,0.06098637188879192,-0.14725999406511922,-0.5324370148399797,0.07487096592532153,-0.31339122558381066,-0.3701885433539276,-0.3568337064213453,0.2526519135818172,-0.30115120274890295,-0.4533404799864505,-0.15458007039918142,0.2601407344531374,-0.04687048316505994,0.00873584533915845,0.12597357021052824,-0.003990085712387456,-0.11990341675846371,0.10235453107099907,0.14913847514874112,0.16225111804953166,-0.029830092090069933,0.039141993050738906,-0.10569835429100169,-0.04112417713433336,-0.06932028045866374,0.11564649782773619,-0.17807956863349975,-0.210267116420824,-0.14584561360924517,-0.07934685976556004,-0.006580607428415558,-0.12175825099407839,-0.02657484582972688,0.0419093136725815,-0.2270036839091662,0.015044741932436413,0.03952130597333358,-0.08141980304831921,-0.08477356723297785],[-0.09044932765596814,-1.2091302803094792,0.4729689263354695,0.12716325311604518,0.17752128970860395,-0.03877043044503277,-0.1407662082145212,-0.4797969623170859,0.22536467397171628,0.09700997503109691,0.17491870843671442,-0.4007161550238866,-0.09092846171801393,0.0408707429360234,-0.061402182505781396,-0.4283840054014095,-0.37170940086998705,0.016963822647346194,-0.3270158209135636,-0.3757304104483934,-0.2011266429204215,0.021592743641159266,-0.21224218308726592,-0.06225341496798263,-0.20149665414761225,-0.020177997657059677,-0.2756664375401891,-0.0987968634089237,-0.03109305413435352,-0.05169299024040411,-0.19663145328407777,-0.1375385205812851,0.02168926489108037,-0.10058037941685742,-0.16334574350983438,0.07173718439892073,-0.2488560790053645,-0.08235678506282831,-0.3077965487539446,0.004251775240897867,-0.1851379512003248,0.07034831734706554,-0.15856913158124367,0.008355457144701596,-0.1980991367368871,-0.04222623612294761,-0.13087884156769186,-0.050818852083087264,-0.16041487353799158],[2.751556969392645,-0.224314440199304,-0.20936436710872666,-0.030879137994444934,-0.13428804313253728,0.03601482744783848,-0.05302021077597077,0.07622560423306246,-0.012697320540972926,0.06972743660035391,0.047351724343526636,0.05120494375775383,-0.0005740870122469474,0.039926446550769025,-0.0027493336195946573,0.011559566427435379,-0.022510485039879607,0.035048332749641524,-0.02400014110029641,-0.006599696645550584,-0.0459361598702522,-0.02108532248867801,-0.05141750382794629,0.004512289576038122,-0.05926429470564289,-0.0034861922896729063,-0.07039922083666249,-0.009447945471155801,-0.05925810736148863,-0.012441166976312824,-0.038116847478784396,-0.02739759184986362,-0.0527519247274929,-0.014920897313822658,-0.05246176660493092,-0.03295816931558304,-0.045101003472817834,-0.0123435588125982,-0.05693293318998119,-0.0012953679831973135,-0.058047586437138574,-0.02894897963375531,-0.026494926644258003,-0.01284907852603625,-0.011062576778783775,-0.015936496643474594,-0.0065166718884401885,-0.016921642215701967,-0.008754950798020586],[13.136378727734256,2.667103849282794,-2.363129628688708,-0.648728832424745,1.6342705382642995,-0.1845203547297349,1.4457689891731595,-0.07416904300767854,1.0292430311869474,-0.17041340574202685,0.9087824119768192,0.40958704221462167,0.3294740155545364,-0.07542964957142824,0.08503027028772056,-0.2323598266107413,0.4173787369091374,-0.4529270942841662,0.2699711491813998,-0.09659369409219255,0.234426421519516,0.6960776097554047,0.2564496343506228,0.03393017918799892,-0.021920156155305598,0.20363494215213365,0.2801807865489421,-0.01883027296714774,0.12953475370176204,-0.2449911690159152,0.062443657851145244,-0.06463408587367236,0.037813147241319095,-0.0727128192241014,0.02912110714304103,-0.015352181084793326,0.10219374063440281,-0.12085620349640527,0.2446072431534433,-0.07611924374907314,-0.05969421454533841,-0.20192703920207078,0.11638842032735817,-0.07826950179304765,-0.1202487773843825,0.06303429422820955,0.032970533577711264,0.07491365097538641,-0.06294773947368797],[4.32662374148504,0.5001674280400793,-1.2100721640113663,-0.0395944185127123,0.1288749773032082,0.03395143218503211,0.14783581829870293,0.09689168865793041,0.06425927702631924,0.02768822510905659,0.0966723936782239,0.06050516832724993,-0.006406392327721702,0.052341455252788306,-0.04679709273253113,-0.031054494227869316,0.026200107992745404,-0.049457622961428596,0.017886460328210518,-0.003569894673356824,-0.01901927907272682,0.09612020018158175,-0.011496328328793432,-0.002856089773813054,-0.06614372248679959,0.029039261319287746,-0.015821551231318152,-0.002427219038216055,-0.0351337781788718,-0.016851376726725906,-0.032100534476696106,0.005556157429445679,-0.03981607183618028,0.0017235142254487027,-0.037954805157488385,0.009894073149397943,-0.02762575869988829,-0.04199772376670282,0.006959882656299116,-0.022182368104905443,-0.035271864312528774,-0.04334974068260427,-0.007803405141061939,-0.023214328799916268,-0.028741619197419622,-0.0019235464276254742,0.008888886485545464,-0.014211068254626128,-0.014718556091554718],[1.3353448437120374,-0.13687114924882565,-0.06519237818712471,-0.24587240972484797,-0.23563508276032782,-0.0028152855004343336,-0.17595453846025377,0.05480482592799277,0.07943721331565888,0.07404324150497203,0.13184073616475683,0.09747397760753719,0.20225325142737788,0.07030938499289233,0.2954853665851671,0.20397996935381915,0.13988263689962036,0.16762733687202966,0.2665884816864185,0.13168683016585353,0.08942826458599511,-0.004867324171614215,0.21371442712819042,0.08680212818716082,0.2829866912020137,8.568044752122808e-05,0.2166843558586599,0.04372331393797584,0.15055098445536128,0.10953749167514863,0.22024092690833488,0.14314185473126978,0.09429112635146784,0.1471267754583038,0.19682123795894738,0.07924240735934271,0.09173913198176027,0.0731330974263934,0.056531138757762896,0.04823295590067358,0.06445335273084767,-0.0011506889705844247,0.06825266327668085,0.0974119250342786,0.041419991280475806,0.020011116630313536,0.01601984100681656,0.039387796686334034,0.06513560387926591],[5.270681886053946,5.465539719909857,0.24101856185290013,1.6753332238742322,3.919847274802174,0.011230927866727634,1.6867544909877017,1.6574254935033077,1.960799131253938,-0.36620868629221875,1.8888622522174647,0.6002780615917054,1.9669322290821185,0.9341562118913207,0.9199919692462533,0.18915735131566383,0.9960369179614513,-0.17690266266119584,0.8415132430774094,0.10704930896229868,0.5476006946563939,0.7968921137835859,0.3528417667308472,0.15260475221814632,-0.3774380603278006,0.7883919488938181,-0.1319160833829308,0.09338788894561541,-0.2498624155331532,0.12089488543955706,-0.18950297816842385,0.2515481104457282,-0.29314585079670336,0.48161097595511054,-0.5335123283365915,0.06516069990861519,-0.0722423633634538,-0.2299619913369841,0.29662645440612195,0.04857404379839571,-0.3373567621110364,-0.11925632815005914,-0.3721393149014794,-0.16730898207209183,-0.37188415408868253,0.055744779297138,-0.296241418911704,-0.11045270650901146,-0.2931035890671739],[-1.1618017692457412,0.1385027627824729,0.9689782267252374,0.387863193684454,-0.14079558683017762,0.0325190763775923,0.1384228126749652,0.02662680562779468,0.14311069471843615,-0.018451364335269706,0.09408193764126414,0.011897435663952766,-0.028769299564282,-0.03868866976446477,0.039863691884408144,0.026970057671952843,0.03950867445284592,0.01716838046588748,0.02603513495378216,-0.004967176293157698,0.04141045193083591,0.07467135016521216,0.05206036213898623,-0.004541932139839712,0.011685906889257112,0.01997895906434618,0.06296756040858859,-0.004854600843071553,0.02998746239812167,-0.015635712945310347,-0.04041966945693811,-0.00027605686219480806,0.06602072370030569,-0.007749555287336169,0.019609582015499668,0.01963467931254238,0.009405314479414409,0.026707890659890716,-0.012673269635445008,0.02093545291173926,0.03538780403582182,0.010853579722387116,-0.006201334117659598,-0.01917332944578677,-0.013698703217814336,0.03701345047431308,-0.005055277468175388,0.029377288045192115,-0.027831441225523616],[1.4725905018149514,0.00730482493605291,0.13806176214056948,0.09268232214558865,-0.0048955515326619395,0.006757820767702916,0.21650861716185874,-0.11224687447015548,0.03949712830963732,0.007553474793509755,0.006280412410747867,0.006209398013247677,0.038227740462727366,-0.020547445685945544,0.056114244376746114,-0.003805873130182258,0.07329785413218869,0.01986577553018675,0.049741367961359285,-0.003804935033644987,0.07426691038412378,-0.008603109330708673,0.02976872015090749,0.005048371423784912,0.07258098130083007,-0.022517140837417533,0.06362930762593476,0.0057121556006486,0.07685314369345428,-0.021496310992857,0.04827049547077767,0.020932027262073643,0.06135808588230694,0.012057911582202211,0.05019717172529652,-0.0014521197095165698,0.027914879457581823,0.031454767983719246,0.049439978191842156,0.011572721061368503,0.044491403347221535,0.0168846425354636,0.045091056252346245,0.01926924168369832,0.036984393879518525,0.01689282590053196,0.02909569989883775,0.013942933998733262,0.002759496510394048],[0.5689469728672807,1.4846644252958285,2.4695510542898993,0.35594687940683006,-1.4150519354935944,-0.13673743379458908,-0.38468512619451983,-0.24918219234018424,-0.05660737096865115,0.2918961031691755,-0.08483512726781754,-0.10845882798294679,-0.11294998228284697,-0.0743117584813544,-0.11502225288003601,-0.015406710452224264,-0.20762094285957805,-0.10973678446881006,-0.03201370576799902,0.11721889273960498,-0.18822773412246283,0.11857917824838662,-0.010008606165195598,-0.0992335861346504,-0.07890472953734608,0.09796404206877464,-0.1746359524201777,-0.08712546836100604,0.042481766122384045,-0.00704849226378864,-0.22888294222111433,0.06380855242838378,0.12076921177948863,0.07592857822608953,-0.13378760731042336,-0.015026922063570357,-0.09102571158179197,-0.05094301589080909,-0.08331305012284614,-0.019176912895220347,-0.038015597827165605,0.10407487222633974,-0.04487369650802493,-0.0820841256388764,-0.09230277655952299,0.0490952184519042,0.036113000918101434,0.0675404060560474,-0.11624598686117864],[9.800456368711377,-0.9468574777059066,-4.712041393060577,0.24180698763518027,-0.8012426357990752,-0.2252462287778534,-0.9913995465750125,-0.17451307410372463,-0.5702893391664217,-0.4128160169869284,-0.45927869546701977,-0.19594475293143987,-0.5020239956065418,-0.07412604380344194,-0.4683917224584964,-0.13291265074443903,-0.2756178893940221,-0.3712896637471416,-0.20459646446877786,-0.13190706494365173,-0.23804830135870567,-0.051116373861704714,-0.2023300008045179,-0.13887513984230376,-0.3196004420508492,0.08438184102343804,-0.15540949044689356,-0.11506826753568022,-0.19425903819302395,-0.025781638268322304,-0.1993166415327667,-0.2012570700342881,-0.09347150514576945,-0.12827999973871249,-0.031223422052517956,0.02914991776709702,-0.1377005043126897,-0.15651498131856592,-0.06704521978070592,-0.04273378440906051,-0.10745124685759154,-0.08644127458175896,-0.04267360478867413,-0.12487731777983042,-0.0779947656459493,-0.048537988884107874,0.011229052095117797,-0.08762334379972364,-0.02753112369340956],[-0.93331946518849,1.5752347992799853,0.18292156969173282,0.4954208835761772,-0.5885645742239202,0.1982185430129251,-0.20766109769915578,0.18992109992525533,-0.2362919625064977,0.1014234725834975,-0.16837114080980772,0.18805417386257844,-0.05158020087444283,-0.026732985854068744,-0.044533561429800375,0.03499375756160579,0.04341724665811475,-0.0014077734163294138,0.029238369908595782,0.19501173094221916,-0.054989215178284735,0.001019553241425235,0.05052577276195188,0.041965610344478994,0.07248782583373463,0.016854627978946535,0.014660979957049956,0.06451550218068322,0.06542372888467966,0.058453092583118794,0.051222647791729664,0.08578715033760084,0.01390884317422343,0.033095892780481334,-0.002126097861956706,0.008654627997650878,-0.004786601590129628,-0.05279794451923118,0.11159694955831347,0.017322322094654865,0.0428989597238725,0.030860057925573846,-0.01042959764436806,-0.014920376426894044,0.04070221248219481,-0.002462137154988407,0.11268685409852001,0.007418516361503101,-0.03398582789786624],[-10.438913286540615,2.6614113867556988,2.3483495263121776,0.4301544381078251,-0.5603433168435926,-0.13389640825384358,-0.0025814327930142105,-0.033765459124628236,0.06777159443285515,0.25593182335967646,-0.1053658561994689,-0.08130499210568437,0.4957026606180016,-0.013047399377304371,-0.22556583160947127,-0.24943632475195185,0.0064576203208956216,0.0406457172595865,-0.21499300705625085,-0.05877798332078606,-0.025529753397403602,0.17327556458067978,-0.00018927269818024359,-0.07469392743213542,-0.14067245218409624,0.16300168443291974,-0.3604561216649622,0.0613335153412059,0.01287228343885152,0.04330035270200758,-0.14020109982428466,0.02122743140706987,-0.0045648961429104505,0.2402278844928533,-0.41075045348274236,0.1167281036658097,-0.20133936810012912,-0.14095632049729973,0.05611120298164035,-0.0066073144344573325,-0.11308308070954701,0.24678890644628695,-0.11521435361830736,-0.11329236722502695,-0.10537437598135943,-0.0020016280668357714,0.018560909725489974,0.04833868214334159,-0.215426022358717],[7.832870362200589,-5.121575908819108,-4.611892770193711,0.944978481807768,-0.8147793305417396,-0.7652014627894858,-1.964982269178564,-0.7935434296808772,-0.6686247424102568,-0.5889653438098517,-0.850846436137133,-0.2850112301215882,-0.5964369120608886,0.13139859970663997,-0.5331434428990897,-0.42483754796032314,-0.7748306085979162,-0.170694890295953,-0.3209725328038152,-0.5091111911767543,-0.6025995833728202,-0.3197096402973251,-0.272229772229246,-0.24414481803958415,-0.31072677646704233,0.08972904088207433,-0.47247366539229657,-0.2741332838806427,-0.19976862725234437,0.12244212815646861,-0.10367848528927157,-0.28050451394978254,0.05160301139310407,-0.1278683864600944,-0.1423970144023485,0.05589921235721018,-0.33113656021092486,-0.24589973528340783,-0.4213928064860572,-0.04946380632720363,-0.4726860744979171,-0.12017276872679022,-0.21143269009493468,-0.01241453019434223,-0.20074804544624394,-0.09731342849601304,-0.11966287987283863,-0.14929466707331468,-0.17450342557227183],[-4.543757832028604,0.9932734501606321,1.3346672590313928,0.169447355988241,-0.5794887821653025,-0.15524798753035424,-0.053411238015286674,0.022977356302140057,-0.15134115653687566,0.17889230419917615,-0.16582365557483972,-0.0007168227747791137,-0.1656771819662422,0.0006990083183663526,-0.12998200404310628,-0.03313820729072666,-0.27377673557076876,0.0460759191389806,-0.09012614294250415,0.07963509884909219,-0.24632301549285798,0.028014088270769483,-0.14232331231720954,-0.016091029137666312,-0.12631121324123465,0.06429792289997233,-0.1886874759066667,0.0003734218515606276,-0.11308893746973095,0.01221120178341135,-0.18122788488268557,0.09236706842195616,-0.1083198722320645,0.04599957889331902,-0.17141383428736254,-0.011807169977959539,-0.08697177302127157,0.02746691922035916,-0.11864042550817396,-0.03741599561049821,-0.10191999028118484,0.01915863314923632,-0.07440433833969956,-0.005612706392215775,-0.08853795042555447,0.013933864562750597,-0.09198643833713752,0.026271701399045957,-0.07096178190634264],[-4.184970210898763,1.7916920397761176,1.404166684025455,-0.2748178224558997,0.18608802932540852,0.021619972167217675,-0.015760052612091735,0.06969057333873908,0.029812361110737137,0.12704583341685333,0.11721458333660792,0.05377693432540604,-0.07423632159904754,0.11365345700320215,-0.01855966564749649,-0.22518162007875955,-0.07111271141762904,-0.07177658238237097,0.09916712392538843,-0.17329812977975165,-0.0516449941108645,0.0486163451433935,-0.06599961747404802,0.03416447489097013,0.03356753898626601,0.08356961071780918,-0.0032737994688866586,-0.033007667725449386,0.036508938257765175,0.03989976402629059,0.15200199260761568,0.039505595500841595,0.0850724748358783,-0.00821614959089172,-0.0672321615260054,0.040009208802882064,-0.06946003430332728,-0.15483027554689197,-0.10682813799372218,-0.051746087971140525,-0.11461454876095335,-0.03662492393409028,-0.02109903170450322,0.04826637498984121,-0.05893456632159266,0.019479262011023377,0.04076070518757623,-0.02577938692859837,-0.028671324557669735],[4.101336881593632,-0.018967027719679976,0.28842509566120217,-0.10354954846655674,-0.37603550003346115,-0.0890506491037627,-0.27750900687769,0.023633105353755943,-0.08442837485888824,0.007721332208405309,-0.04321461738491873,-0.013044634719013593,0.05137515219450395,-0.026173244601437065,-0.011038388866226198,0.038831365713857834,-0.005206167346999945,-0.04302680592936077,-0.04197985543261132,-0.00828922769559116,0.01564442357831369,-0.08601269764861597,-0.0026985603070598485,-0.012645489804468113,0.013595599929881349,-0.007195347551386079,-0.048672963501895365,-0.014974990007281941,-0.009717866451519213,0.03532313789238504,0.00816234861347245,-0.02025855038664504,-0.011427767022918495,0.01375636842509728,-0.011517428178988513,-0.038147764263538164,0.030226471836907708,-0.04473627915258351,0.011380587103797071,0.007804196648170185,0.0038159955572118977,0.026979534353825812,-0.008910516975331476,-0.02013776098931656,0.027737886368289718,0.0004803864024603594,-0.008820884685144222,-0.013223761786230028,-0.007134552629704795],[4.640426677781926,3.8441952778192903,1.1495128198571403,0.2832386672473419,2.500514400341461,0.7021858520244043,2.2611895272738916,1.8586757558903466,0.7466432661109047,-0.43186958742174075,0.8187306937643715,0.4765964561012134,1.1676205606056977,0.46819318385004594,0.1842963818320125,1.0352963131319404,1.5606232505305322,-0.43231726804028703,0.4007960259615802,0.8212131536224593,1.0924316563542877,0.4954388941180007,0.46615864562436515,0.4974462020207455,0.14619509746579162,0.16685187463229403,0.6425432312664835,0.4785155479490877,-0.007599260411938664,0.01964286829009062,-0.11053463063009461,0.18010549168378026,-0.29887183590364425,0.13000198294331658,0.07022838318188543,-0.2133079730829701,0.5777988682535998,0.1402468141428266,0.9815595516217107,0.17862027869249977,0.6965391491627446,-0.0811482659140261,0.2923846669290209,-0.2210231078821714,0.2102091840642928,0.21515183032864443,0.09761016595158489,0.031360954507156556,0.09527772401477096],[2.082105571233034,-0.1671853838396732,-0.35795428757406217,0.16915751390524586,0.10257935200496661,0.21950503723385256,-0.04382621476448508,-0.18146986371897872,0.1806100371907601,-0.12100434490538778,0.16802476766254468,-0.17169566969652936,0.1904769323892848,0.03093697273248619,0.17009818191458892,-0.149652359142801,0.2640361863639223,-0.16532677728844009,0.06886732867457976,-0.22880862628542764,0.24766831249395335,-0.07614406948710027,0.16996013199431229,-0.06530566742513942,0.118581181655857,-0.11327146618855992,0.14866215872836006,-0.06313812219015717,0.22716631660755238,-0.044944855386911835,0.25809053791125086,-0.20809313055393588,0.266775733422222,-0.08534271849439017,0.16130114060171116,-0.019690106750008225,0.07360806951102669,-0.13209517391698047,0.1358852541130978,0.025066688940667974,0.10679380457823007,0.05473329313634972,0.09978263122302918,-0.024797557560418335,0.15226185370996553,-0.03209912017843022,0.18118268980079857,-0.005299656082843985,0.025659642168618525],[-2.9338540231814214,-0.5571750858262755,0.21987855578465476,0.052741583891633824,0.04901843129300239,0.012249716519391619,0.15887470713241503,-0.06287935652422365,0.013661084691920407,-0.05353043092713342,-0.0464794965827043,-0.05773914983754384,0.016523242731305177,-0.03116178354806677,0.03676674759362047,0.039577824625358984,0.019846596509791703,0.04558796666852043,-0.027830220974410576,0.03623503269113648,0.016147203316533725,-0.035346520231091964,-0.007237200263267336,-0.013278390517761411,0.03304273575280531,-0.0512779149832804,0.029803918217185147,0.021035646190615702,0.044291784936570884,-0.000691242817319865,-0.010945605319327208,-0.024443848286289625,-0.0026743940070039695,0.011011577568639288,0.03610135149964144,0.0029276435144593906,0.033342301327707075,0.04061239809220779,0.04255301405313522,0.014474492504175146,0.031018185261065284,0.022757205174419468,0.020260693484200426,0.01597390356019436,0.025419676723797177,-0.013596346307707351,-0.00422144887832814,0.018530058771649458,-0.0009274054028829389],[-33.546114084922046,0.22031385971617032,9.187506583328583,2.1558713064137796,0.7502378653474989,-0.47959282076491844,1.5081971075138383,0.0844994736494294,-0.170118356360174,-0.8658239917772463,-0.8767266299908268,0.44397089259044736,0.5403036361136685,0.08968774815358002,-0.49501216341224047,-0.1873511801443985,-0.06600601191719842,-0.6239179923385765,0.24305423271101637,-0.004868785448046142,0.5289896751892601,-0.20373180888800546,-0.004419293307347013,-0.06130595683427242,0.4376654110891702,0.06541470417641244,-0.05440261965668407,-0.20635332750122007,0.2160313890799606,-0.011456290558789725,0.11496808646970105,-0.4207274032989945,0.5087243902082018,0.007926672962397923,-0.22654244636929827,-0.2226837115323994,0.5022441104363784,-0.11341426141143628,0.6129284081616225,-0.029092740444675323,0.08249884988822821,0.22519793903937274,0.4207094217865721,-0.0911985125613091,0.1428089153496652,-0.025836078770428046,0.05028499007763632,0.1431470341713968,-0.10592173127962969],[2.692253632833775,0.9751550245765946,-0.7812355481159394,0.25214172322179335,-0.4452325346388856,0.14031064528572906,-0.2505575832565183,0.16732343619970663,-0.1876068477435788,0.046544227498479855,-0.08922374546957904,0.11898856476986902,-0.021442003969733097,0.005962122178612587,-0.021388133960284637,0.058750106278980266,0.05627120754985518,-0.011176698801600661,0.04495751795043614,0.07906754189657225,-0.005709952262454199,-0.005116857731350895,0.038490389819725045,0.018000599065805565,0.03766232332793878,-0.007967051689230167,0.03682183066974986,0.025702735518245947,0.009395057532766484,0.029642421157075348,0.04749459669830897,0.04508860092320986,0.00918015870844312,0.016311919902773954,0.02688116833167216,-0.008189917494871026,0.034553242575308625,-0.02742766198815675,0.07938009688236378,0.009566918979547916,0.035643403720743,-0.022093566057725115,0.02855042144562082,-0.021660123657928517,0.041816464964082346,-0.01494874342612474,0.05295955669921284,-0.005501717704122045,0.0176171981830174],[4.342924655983023,-0.8016789601024777,-0.6245622884556732,0.499855468551622,0.3296598529452492,-0.0060035160659525745,0.21100132728530732,-0.06781654826620871,0.1406641748304285,-0.14588029592617133,0.10319863506173377,0.08052995471182073,0.23742474355409607,0.049999547768636465,0.11407894033783911,-0.08151374638926856,0.17648036161335318,-0.0003244501037439545,0.18903936571740532,-0.029569829387203518,0.21187722022838243,-0.016926970221330356,0.1144643708687754,0.03266756955750904,0.130044696585732,0.0428810567313719,0.10164004460400945,-0.002366102560499599,0.15355339609423985,-0.003320468033650826,0.135448408698612,0.026950705991506445,0.16727573647951682,0.060677995786855565,0.0905891178261693,0.015637316037613543,0.07422483014204774,0.0021197060457245247,0.1527503590190231,0.048263587168464295,0.06349584015568137,0.026246881499853925,0.03808037008229531,0.031186148384465387,0.039130482900017395,0.02999437384067781,0.050922674817473666,0.0020615769477990574,-0.020320206202962186],[16.299181815528424,0.5622082471601699,1.412105492929394,-0.21634403348005454,-1.3156131803109632,-0.41524093545026114,0.3033436879871781,-0.27995824324183466,0.5314680813057128,-0.32331758568021424,0.4044852631413962,0.1765423807315054,0.23189099465218685,-0.49218780529508743,0.004809820536072042,0.19313559037785594,-0.0003587743500445005,-0.5165359784541509,-0.011783769750956142,0.15486998025904228,0.29499247759165337,0.059502120471271285,0.07120737450303849,-0.10035459162249147,-0.07401162210859258,0.0598589086369351,0.058537986654800464,-0.11337930595697567,0.08585943795563454,-0.16315693632317635,-0.24090908939446917,-0.3695365165813324,0.10333273817132452,-0.004113756232511055,0.10476071335356399,-0.17761744814878958,0.3617654485104617,0.12342447551455984,0.23964402862284118,-0.12077927495052108,0.010898112959583517,0.1414246581535772,0.1341382749256535,-0.10250225805375636,0.1457940710938355,0.03384139254297545,-0.030504153818682635,0.2383461072684235,-0.04026686177056511],[-5.659619720062685,0.46253706216818513,1.0061866079966546,0.21306729059005464,0.25548561362399674,-0.029754568274976013,0.013053552424038054,0.11047084246123386,-0.032291376642074994,-0.048507845875506064,-0.1441643204451582,0.11015824404842486,0.0527853127152789,0.04865046628636955,-0.059774481732142115,0.10491697930959606,0.04575542609719242,0.025286091774096756,-0.042153088185882266,0.05625929525510702,0.005562937451930657,-0.04903308774884964,0.04641846090637836,0.022397153017887878,0.07047159461157707,-0.08831221367308019,0.012319117792109615,0.02224957723557273,0.0011339845166132084,0.08664843415057738,0.0020144531712616006,-0.01502195461368141,-0.030654374366334922,-0.03240197766790053,-0.0302892605822307,-0.06058740519406999,0.08066408934328448,-0.0012309387220265078,0.07179140192614304,0.01509347614677225,0.021688218253664505,-0.006122425156636635,0.019057747457745168,0.008112830352623703,0.027088649503809768,0.03660125229363112,-0.022160041645983404,0.0024418646401248795,-0.006834920687789296],[-8.536263846218246,4.648856231055151,2.0810344081402676,-1.7137558897420884,-0.07132986301211186,0.08146882546400151,0.3513717055952147,-0.38210942835244005,0.030684923386703825,0.6159475402020043,0.22277795646265508,-0.39313531465465135,-0.02529346735624191,-0.22603936295877,0.15540047924382733,-0.3365783844923761,0.07052495416000075,0.039474177476204034,-0.048364716570502744,0.043500879682963255,-0.09311078020311937,0.0957599092237179,-0.22617996341688254,-0.05420156815275968,-0.04136712130014301,-0.13489663623074982,-0.06182767644468696,0.08889010149399375,0.23761599153028284,0.04312481118793851,0.03579062298842974,0.06807368462053622,-0.05901296868543722,0.12605003710463425,-0.04445859784784376,0.06759683245414227,-0.024578380053065136,-0.10605716790481791,0.1795976090563264,-0.09274732065241481,0.010350425253298661,0.2008715433468137,-0.028462557198626098,0.10446611070657812,3.393062217837283e-05,-0.016092759656402507,0.0680744238147273,0.06217533366212705,-0.023152721102230673],[-17.125676055102097,-1.7125964759024126,0.44522523470301073,-0.9269559042118395,-4.683474880142994,-1.1143681751091812,-2.5754491046738233,-1.3855265984479446,-2.9344965653042956,-0.3579733454897853,-2.7093882236591,-0.8970574398277756,-2.522994772665367,-0.7655702744834224,-1.4286556042571663,-0.95713492154984,-1.6922270852968235,-0.26590399955420124,-0.8459679919340952,-0.3169496362067699,-0.9931229817129361,-0.9145906746197869,-0.6050023430984005,-0.158535384637339,0.03983500226974785,-0.6603838734526667,-0.04850654509111986,-0.21372754715816483,0.20300686211964442,-0.3749095137630633,0.16340201419638534,-0.4312361234189412,0.11684233925039941,-0.3635743134836623,0.39578329517322003,-0.08932997478716934,0.20861556056951785,0.12402618558742909,-0.04219738039392333,-0.08019292595044471,0.26545789815515203,0.05507280929030775,0.2617082860581555,0.1574992498144745,0.47834083537729255,-0.09021349276488083,0.3462874508588383,0.05367526222850447,0.2489575066008132],[-12.49298466651731,1.9440509150797447,1.4744700820946226,-1.1913745446787152,0.15611781561196,-0.19155607370532488,-0.2907939243517202,-0.7283780164877784,-0.3237944899533246,0.33792830310824123,-0.4470787751304154,-0.5583135405431309,0.19330750600852717,0.06936614785089233,-0.07210140519174725,-0.6507956080348564,-0.3194045216005134,-0.06378827255542238,-0.03627759800438007,-0.012246729538532683,-0.3538056687927855,0.010107936458353233,-0.4179660071432346,-0.17882305050939326,-0.21633341971250503,-0.046496520036602734,-0.3846925787231309,-0.11711916034217966,0.27434456169724075,-0.09543494348512496,-0.05813075623369612,-0.023022290102133934,0.13107324310855326,0.24504778663094715,-0.277303939577359,0.11374344700781266,-0.21870600229246978,-0.25815514343857715,-0.031902842616087876,-0.27486932543355935,-0.25097109256896694,0.22831019961622573,-0.11280128572014851,0.1643838931708057,-0.19822544638207854,-0.07880132172708802,0.09026810250854286,-0.013207100499511936,-0.04708482473393561],[9.620200471681448,-3.3798325119340413,-5.43942033592551,2.479921113983598,1.7579055545579239,0.2867075103463823,-0.9249736826956672,0.5315351220454813,0.03927600599254862,-0.7667930040724632,-0.29200991490238715,0.058756426903763606,0.5823878727464721,0.5687484228225222,-0.03958202014430259,-0.1056537710493147,0.18017762906866439,0.03167947487704101,-0.42224590357225217,-0.21396692509649332,0.3433299186284644,0.02802050534439601,0.054238893919370826,-0.22693701077629314,-0.18943870982431865,0.4907649552754073,-0.09431447822298047,0.035830087030364326,-0.2651680549308621,0.0762107975718244,-0.12259666796899307,-0.1996296269464603,-0.10927729709520147,0.01743464851195396,-0.4052085945072384,-0.13449243064128,-0.05847331544316726,-0.0013262312262210276,-0.027336194195940317,-0.0004962580305253322,-0.10831095976578216,-0.05166962274815357,0.033759488174372326,-0.1430526189211996,-0.3648529334465547,0.03939658462965014,-0.2506977537606881,-0.03152618348325565,-0.1366741726601462],[-0.2605451774420388,0.1701161436592038,-0.20360851049838816,0.05780083322830332,0.07958997270663663,0.06396548622941274,0.05791811324141823,0.08983347440130071,-0.0030556241077657484,0.03401238851505728,-0.022546180378065075,0.030265880582699144,0.02198682711803034,0.05706561490609243,0.014711909338761884,0.03871676839422344,0.02452169191295764,0.0666809847112999,-0.043287145490437044,0.05677437124996433,-0.000883541099800446,0.0174422617053043,-0.02080916269866447,0.020696481153724654,-0.021061666827864662,0.0017088927248329333,0.011876696479884164,0.031512076715771116,-0.01833865141839651,-0.004799250704028248,-0.014819402570666043,0.004423631427251051,-0.02485551859454106,0.020172953437041327,-0.02195322876956212,0.0044170395706180764,-0.011118044207540052,0.03888033957864577,-0.012961048323564782,0.013145590062582934,0.003456588568485417,0.006976303995449331,-0.018698927621729727,0.013240290816581103,-0.001237653183112029,0.011784190567644386,-0.004868558418076196,0.007554168224409857,-0.00292006322368361],[-2.6299592133861154,0.7991096315549899,1.0297371377998927,-0.3110759401483766,0.004156895482946074,0.06381163952009059,0.0910608398727129,0.08810710689412143,0.12590723447321256,0.14816924138698748,0.16749831770840898,0.056917024213803465,0.14531416710968453,0.03659217931051605,0.12105083062609226,0.060788393509291085,0.12283078237141945,0.05725182699312709,0.10388540606154738,0.04521207808190158,0.058734342262518946,0.034081579112743315,0.050676435069827636,0.043712659299640244,0.06459206696315149,0.0010516924294365873,0.04495354027837214,0.023118679193664903,0.061360612066359115,0.05305143636497306,0.06934157776102338,0.06158443715354587,0.029968652310098067,0.07276314821563101,0.018608962050022826,0.03447195894279881,0.012398269294312477,-0.0067817915385180805,0.03538186931314441,0.028378154091269186,-0.001840605761609088,0.021134040885949324,0.014175209857955085,0.027422274975233897,0.008547919462691945,0.01162421044567308,0.01051268149283651,0.015940459156451182,0.004237207647965574],[1.436498500341863,0.06689007112169659,1.0649065347164999,0.30303426837618774,-0.40970812261194106,0.09989976199371557,0.015534089504535093,0.1756573897973833,0.05997061913274788,0.035619982763830965,0.11134161856419243,0.03789253047702649,0.06980635984479368,0.056032933279604825,0.016226980753095317,0.18825527438926382,0.04694202329705093,0.046811859212834844,-0.012898954670985464,0.08934688516697528,0.03771192572400374,-0.03748173268351893,0.06983864232590031,0.06708979356711764,-0.017302960484723025,-0.01745792109275202,-0.015729974628241057,-0.011775289059964177,-0.0017322935182175472,-0.016515659875510527,-0.01667908872556125,-0.0058659245995644075,-0.009470449019599442,0.041943574126948435,0.014655413299099198,-0.024960282243632354,-0.007356151276660281,0.06913697901252665,-0.026376574125084325,0.04059238025659591,0.0389360206277978,0.0038388476159114163,0.008894251559784027,0.003102875422165973,0.0482827742474012,0.01840832079552587,-0.01534633164960202,0.012986940223457462,0.01114961560349408],[0.7078632526224167,-0.12767635390033685,-0.30057025417428407,0.09142533154359186,0.1548026919399058,0.034488315290957514,-0.13969354183889177,0.0770481796731904,-0.0036112660488375635,-0.04791415327697156,-0.022633012369107235,-0.03276848275183611,0.05963281106025388,0.08180336775083744,0.0243221753803741,-0.010299356423196045,0.006316221465833134,0.04717404394774906,-0.02508150152159095,-0.009236203297765446,0.006062159209978072,-0.015163636256158035,0.025170897938254784,0.002760286035066218,-0.01646052741575367,0.03400613205582596,-0.002604113019771305,0.019446117232159405,-0.001057066678591537,0.020152611684203108,0.01044478602162804,0.0026966374823580125,-0.01738147601895898,0.037176357096671116,-0.028787031056466482,0.012982247003501441,-0.016381732362619396,-0.008249260294250577,-0.005132827284787942,0.004930555301470662,-0.01398410789419598,0.0012310798594974872,-0.014848458735197662,0.008943494034402972,-0.023415543065914035,0.00593092520043578,-0.0008075250290938739,-0.007966463450411666,-0.014600573680006815],[-3.6648606308042297,-0.3465636078107086,1.212366899485788,0.13539755319176686,0.1955862623510659,0.02636963435517152,0.07493499001580747,-0.008881238212315533,0.10065570769296216,-0.020740070255442302,-0.009079219539574896,-0.06691572494321886,0.06174610208696577,0.04904409290269595,0.023997717315660215,-0.04660920229553808,-0.014186057313032043,0.02544386040548075,-0.09070367559217117,-0.047095770735344135,0.005106954801068403,-0.044902394060965065,-0.022911147403450013,0.0058760836353942605,-0.00834609423986723,-0.03302646525026995,-0.038105501110486696,-0.01323611982025662,0.015032122253462468,-0.013865985159163317,-0.010857177868167176,-0.08270181309661881,0.028322458114167875,-0.04332647956346259,-0.028315593420017195,-0.03842238476599761,-0.03051659801171767,-0.022011693953729785,-0.039856638280526935,0.0023243554565446542,0.015920662727456406,0.016700033572917168,-0.010595427547735608,-0.009645384237698727,0.0037822787690378924,-0.017338321683128002,0.00828560667018578,-0.005241207020880269,-0.019943106585251415],[0.1540182909191672,0.3969541595559122,-0.3165573310088155,0.28856436768007937,0.48960333964484287,0.11252193570600336,0.39342510298712413,0.13620866787527816,0.28896923555675674,-0.05642541549786207,0.26356673317133483,0.19845538699977328,0.3455410051731721,0.004537910788077982,0.29077679640378545,0.17037499022607477,0.3320034945978999,0.026686919235678296,0.3497385225817902,0.19958317120247338,0.30788360674260107,0.10440468665962453,0.33655427914502556,0.08631702128227925,0.3322775294902335,0.06516778816166822,0.33345457500085646,0.09426266584800548,0.23227040770592347,0.08414702678083437,0.2110990102505687,0.17475277713212983,0.15527497224688577,0.10916402385591618,0.22832322144046963,0.09773928419997421,0.1897616926921801,0.10105632337571248,0.21433815708438986,0.05906612672192523,0.18477432913006978,0.06070855627770586,0.07079055771545892,0.04473475268529711,0.054804806187093075,0.056202289428171515,0.027437564168807816,0.0486084450647643,0.027883739848848905],[-15.25847452789981,0.4778503033541027,4.211251753933194,0.0013198407064919742,0.18104240591112114,0.17560467186484358,0.7737120662376213,-0.06157382401165968,0.37299338506590346,0.13436070962928617,0.3062604291773503,-0.2764804731749658,0.2516683058140631,0.14919727398289195,0.17782745340755926,0.023304458119232714,-0.02685552897551227,0.20462932889513105,-0.021971884261824656,-0.03761092433543451,0.11371381571270887,-0.06803008678216046,-0.05155690535964194,0.13572320942874,0.09758243022322105,-0.15451600350725356,-0.039227562306763546,0.0037678704795813586,0.18375528138584554,-0.04330347084893465,-0.03438430074245104,-0.0677488759128894,0.05446023368829693,0.04433000960800726,-0.01333048153415822,0.03516608311965227,-0.07915665498857334,0.05429207019927815,-0.10526593764406823,0.01143819902480602,0.04959427295512112,0.13345637605825422,-0.009212293845956652,0.10574747492302969,-0.026018013974369234,0.03563335963454707,-0.059435021453100416,0.011014491485521794,-0.018568402914787917],[-2.118888660958729,0.27762723427637787,0.2974658758814278,1.7226682880216666,1.1790638540842922,-0.12990166871811457,0.25018444490096486,0.2964890562743687,0.6350708700016753,-0.17561430350767795,0.39530234146149407,0.3499677202592353,0.9546609804097257,0.23755776532163447,0.3375429024674574,-0.027279850210253163,0.49513062926000934,-0.16565272948611443,0.15476491498536726,-0.10728874095313558,0.3304859469405579,0.16350439780387274,0.1797079403801991,-0.15470983084510814,-0.04342900661316941,0.30588725459156507,-0.049212456227432734,-0.01628468753492839,-0.005038350351934704,0.005224509656785314,0.045174094206939255,-0.006689081642566115,0.01146012064759828,0.04131203572258886,-0.28473123432514874,-0.19891971526207247,0.08518048533275668,-0.1164045117676355,0.14605818882246224,-0.009197138592258094,-0.13284147225102508,-0.04604760438267078,0.013100199238849088,-0.11830883325672219,-0.17287977283498226,0.031522637248838324,-0.1704534548390285,-0.016196360942054015,-0.09635995416983967],[-6.972312341062266,2.8713796852158406,3.6733051615097976,-0.9913884845551324,-0.5833357025557261,0.40647113578565713,1.055383585600611,0.12677589689006047,0.6087436903496956,0.6229060579011427,0.9326033052242546,0.07119390860331272,0.45032368089437147,0.10432085412121238,0.7338545260257888,0.18726468353655318,0.7208336138484178,0.24966570164398724,0.5656769176605561,0.007746682568867674,0.38289014068520205,0.02699066231543562,0.24681932769665121,0.22202407086141776,0.3318244719153377,-0.15432492413955698,0.33162647435378173,0.14089420292273638,0.2910789478344351,0.2557595339606426,0.45312150601274037,0.1813739345271111,0.24644617401891622,0.2456354046666819,0.2574553755898705,0.26911458059785104,0.09832866663228507,0.06869383501359033,0.12189104334627315,0.2491986599622703,0.05410390379380126,0.10458888416566582,0.07710183291323006,0.1553225801332205,0.24781508909085637,0.06286982520996644,0.12800431604810283,0.08442180786780376,-0.017172452559122565],[-7.758871978655313,-2.3751922050077865,4.547797136399869,-1.3466630428542214,0.37886195943288653,0.16553678610677486,0.5020654184448665,-0.43591530568617815,0.3658831465260658,0.5417635225069538,0.4049110565224246,0.053715843409949664,-0.28957537682252615,-0.1984395038293698,0.0958129476553009,-0.1794532297289737,-0.20971351561437346,0.26738775956322675,-0.07270471964310857,-0.31028898184567244,-0.17214252586841627,0.08278202710039909,-0.017741238570806263,0.01911693477500215,0.254311958201423,-0.12255850682587754,-0.18305197515382046,-0.00935783808497338,0.23666927468076346,0.09468869811496884,0.11934600271952503,0.15103476067546004,0.08813221684604645,-0.07152607721387805,-0.004689317575714923,0.012644329314046479,-0.20783051568601654,-0.214606450393881,-0.23807055813309114,-0.0615532534386002,-0.041760655099273354,-0.0731312287834942,-0.0482882286309258,0.0804293986990295,-0.11566248183274183,-0.10776307391723317,0.07429784138861281,-0.05833546315870369,-0.06092339303497457],[9.792057002192841,0.22770810532594368,-1.568509366802693,-1.2229516068362183,-0.6824263458162084,-0.23917875000605665,-0.3299467962114109,-0.08613374108767577,-0.022669025861709505,0.16417820035031552,0.042188575698725396,-0.26579268583877297,-0.2899487087946487,-0.008347297705958436,-0.08224632882837579,0.039310936053257174,-0.05938423805285873,0.0035632418046974063,-0.030440563624400136,-0.03811111093178159,-0.10179297934907904,0.11404039417731109,-0.047952858296480015,-0.031416562986873266,-0.07746890468002235,0.015249026066135471,0.049336888173429065,-0.05105613569345381,0.026095580638827245,0.034101733650054485,-0.07203222595970735,0.04336704218652786,0.002770546332920852,0.09090553083458473,0.10054211693831563,0.07508402014191529,0.0007664362839110721,0.0013197855765644947,-0.03171889333427163,0.03610520777375822,0.05756392554265671,-0.032341324119550374,-0.016938980542810487,0.020060096322196762,-0.02603782331243865,0.012783075653916372,0.0005139937178219326,-0.0017945755139754864,0.0035401819026818003],[-26.423811488342448,0.9348645205113183,4.210986030671418,0.17056577443694432,-0.1686141363141533,-0.2497735355021983,0.8780260221093826,-0.1206917212933187,0.4401382013557344,0.008049375185863615,-0.07660475115551803,0.249778609253807,-0.12312151613564566,-0.16227154949068845,-0.34477156995990027,-0.2739288024136363,-0.3248642422147059,-0.303008840768744,-0.07604529733044366,-0.33010237860417724,0.03085493623343668,0.15608633196457788,-0.036524182057280634,-0.2235915229914515,0.1862384481288444,0.0432314862454855,-0.12041908200422316,-0.16618177005026322,0.06103991800391747,0.1255577044213605,-0.05498342206042206,-0.21147320965570365,0.1927320919378124,-0.08561166259878092,-0.22363651209654245,0.0070330213253969745,0.040379936735549016,-0.2392058212098397,0.019708635603065634,-0.12143391644684076,-0.19006680306722873,0.07867762692159577,0.12835707369002702,-0.05769131774262369,-0.1308249059877828,-0.006281507129633084,-0.0017891497659753683,0.09934046497470522,-0.15618699269952446],[20.38383429214122,6.095100455247541,-1.623082315192815,0.6315255997648546,0.7833847335507763,0.6116533429756942,-0.17970493780681354,1.4499010837352673,0.015780017019597827,0.060089098419578164,0.42507078014958233,0.7512577275334137,0.7803245877424203,0.1754012107416775,-0.07154822883440333,0.3579126777398993,0.6932507868456221,-0.05396333035887698,0.10181620549918785,0.40008660155306025,0.08810119930571872,0.2372208803021028,0.18624684880737466,0.3295235308882915,-0.02803784918403048,0.13493184463524335,-0.12993629580713872,0.3965171303979844,-0.14848002928139203,0.23542281956728797,-0.046503144130921596,0.3586355473359407,-0.33635005241377286,0.15668172907897038,-0.26018620448928576,-0.11861779121430291,0.12448139227046814,-0.2900345952713704,0.6462822806881797,0.11880544955330562,0.09191993109661141,-0.0837664187406436,-0.13864794932010782,-0.21989587520606213,0.12042102268145516,-0.09550802868465602,0.14755167798668464,-0.12306092202782931,-0.09765304949521814],[2.913116562191034,-0.32154120312764367,-1.8208302809348496,-0.01817845128537555,-0.2183194849534739,-0.18702216914390069,-0.1702594094654951,-0.1446673600631957,-0.1748243466407135,-0.02552671787096121,-0.08906755299192824,0.040003737359014926,-0.056754488813877645,-0.10461529032841907,-0.14909983306380103,-0.11244302320399402,-0.13862643779117736,-0.07794104216090796,-0.10787643806590914,-0.057183747449019384,-0.07593330562429852,-0.009724452124155004,-0.03636464578504203,-0.065625015097562,-0.05453441340153152,-0.04643548694360407,-0.08990720237574913,-0.014724531816505472,-0.09877384371431165,-0.025530576750263566,-0.0547269689658742,-0.0012128988762244637,-0.09102354140186328,-0.018626642014985293,-0.054479040110427215,-0.03875542871367297,-0.020047426947484525,-0.032503305418860186,-0.004392231941775514,-0.047862340187809806,-0.007196112050003835,0.006069667918580255,-0.027415650286270837,-0.07518146075071909,-0.020943758175684876,-0.042625947866824686,-0.0041713933807417105,-0.009176859711276074,-0.027811861511212805],[-5.519599808887135,-0.9084337297359654,-0.6315260040254963,-0.316909332419298,0.9308940658941826,-0.0864332016888708,0.6659685942265722,-0.24575238286550988,0.18365469920788124,-0.15070581429968513,0.1725857997425533,-0.048475617710863034,-0.1623027996747878,-0.05577548837259263,0.15663168956763815,-0.11080478541190092,-0.012806575357514711,0.015370201469031617,0.0989493033301864,-0.124625255659357,0.05157548885490783,0.084366114562954,0.009222385750059863,-0.024029477431882138,0.0280832989199166,-0.019903316936826107,0.23014835709703838,-0.06282851876794696,-0.005602400730114497,-0.1487217457673393,0.12987081570879466,-0.0620115461756007,0.04661276482572508,-0.09015690813071871,0.14318603936290117,0.022425196302699606,0.027566958646679043,0.11092205018080883,-0.1005415201721505,-0.07077832867996559,0.0027366113059552654,-0.10432456336829475,0.10947797950916469,0.07317473808414618,-0.041071137294667506,0.02735801660150105,-0.0702914229867534,0.041547933914464714,0.10578756127403026],[1.3789453986122995,0.6818790971958022,-0.25666420981502114,0.16101472357487417,-0.23534355483068942,-0.06805457439717878,-0.3476780414399977,0.038644706742549516,-0.18602604912702345,0.10295780343037077,-0.08087800479163193,0.09890412460971161,0.18041007727854125,0.05485727877963775,-0.018589013438074006,0.019289971953322372,0.1593746218193633,-0.0312735482365983,-0.04567141912865869,0.009590513611668502,0.08828602026285885,-0.09318584421623793,0.006740312765469807,-0.004197065715581427,0.06326609480161866,0.05699753592911752,0.007457501411485605,0.026004218151696123,-0.03931733983596412,0.09734933684424792,0.0750160168020454,0.01469365832764293,-0.0076073185232174165,0.027695886527672107,-0.053272975596338236,-0.06929418837921769,0.05522030092964621,-0.04709218305505709,0.09220273931663703,0.09216783921565168,-0.000650240318866103,0.024864158100315083,-0.019819679664352056,-0.04719025942214481,0.06489145011647446,0.01667957125575452,0.010400598275052415,-0.02421727481131211,-0.0196059368890012],[2.4110530480353924,0.08073267637675628,-0.6763683523379324,-0.30266392136623455,-0.2209469197158933,-0.06036302036039828,-0.06707343271489367,0.03521573279945432,-0.06531310309603493,0.0030654814914432755,0.0011929911274243365,-0.009502456614199127,-0.04078065706755031,-0.022859005393238076,-0.030958287561170816,0.028188288555987352,0.02129560371013374,-0.038249627337700653,0.01025963682079917,0.03634346242578022,0.011100010086563422,0.02139533630833321,0.007558047406845052,0.008822889312099754,-0.008124027777668458,-0.005616026274141864,0.03025100229433974,0.01151878585928168,0.0013300312561879252,-0.001346048980599915,-0.008493194460898877,0.018216334288009218,-0.016718640556121437,0.00939338703860964,0.02480394300752044,0.0003597882912594572,0.02266744820228092,-0.009007560941692642,0.03624242975079005,-0.003166235854039207,0.03811183244562642,-0.005006630909693137,0.006949135233591835,-0.004861959430108156,0.013529935342365832,0.006940702493548011,0.02166117469069033,-0.0006323849472606849,0.009744414875901285],[0.7523537551179219,-0.33572700927019317,-0.04887984693371076,-0.19646771699968768,0.03255407875543155,0.005135750492851859,0.035950754086126434,-0.10571320712243164,-0.014159973084388119,0.007809671271114456,-0.06793647214771201,-0.05662292353715595,-0.06806528605387042,-0.03836251123248274,-0.09630891575273713,0.04918108836759816,-0.031967513999999524,-0.037809350104851126,-0.11650312310671068,-0.017634302247138876,0.004081920824242804,0.009943854186818379,-0.0017908476157017987,-0.009189428404845783,0.007153626565468626,-0.02598754434172693,-0.03788536107527456,0.002192461360474542,-0.039191815691733786,0.027809806120631712,-0.05227808043502224,-0.03325089895624134,-0.009794179648550073,-0.06324881176055003,-0.01857431206153941,-0.014184517270512382,0.0005233626837413231,0.00845100366470204,0.01717606539686026,-0.0022763074617536618,0.04812948435355434,0.026840402986739818,0.029795240166135533,-0.04682064673812587,0.0019338497959837002,-0.010250783315193664,-0.002673110328904145,0.021332973509205922,0.0028127080788514698],[6.170142139284933,-1.6691801840456815,-1.134981814050433,-0.2570659988386823,-0.9882887980531337,-0.3148224498995912,-0.08943759761992207,-0.20061554930155964,-0.06335285237957279,-0.1289765485747924,-0.1954707909764814,-0.20521810045276645,0.07289745451353678,-0.21078948552845686,0.07396647034284139,0.5202978238026155,0.1915581811369781,0.06308721241055007,-0.05339717727337674,0.37533681928705515,0.08787796392364854,-0.15399882292605893,0.09783641478712371,-0.03345981754879436,-0.13130486919269513,-0.17967956027623092,0.09358317977949374,-0.016682979575012857,-0.10630615758128013,-0.11880711381317062,-0.2220284789932568,-0.06403833026977285,-0.10220637626648067,0.058741843404261355,0.20226912066947156,-0.05485645974973922,0.16397867536785085,0.35368238315369604,0.11469325702314732,0.11974028163138274,0.21657398981981044,0.07225445798398146,0.058874724116767536,-0.06263163684331334,0.20197341422677573,-0.0018168509276023826,-0.08079858295395423,0.07745980132175043,0.10251164257277262],[-0.809190422430126,0.5126803024407277,0.7031037048659756,-0.30536691505992986,-0.08321527435337386,0.0908574207028421,0.06914604265706886,0.046508828390310314,-0.018385047749554902,0.03989041425633907,0.03221952760007503,-0.03133437366183041,-0.11102478959287873,-0.019989894498550115,-0.016290284751430028,0.015087842993721274,-0.06725826029492737,0.03139460005773196,0.04315042720441461,0.040982867831929184,-0.057585656727062895,0.0011080127742395095,-0.026399208572501633,0.05805976620157275,-0.0021372435961609453,-0.039255265746589316,-0.007794747658240151,0.000688625371041877,0.007643513444298448,0.008087592720594405,-0.0002662408585828123,0.032818229550987815,0.00793727215759711,0.02477290037319726,0.03474306483809828,0.049554126301380313,-0.030929304272324917,0.011586051130962697,-0.02266349988010434,0.005374261065061737,-0.0018987823853016225,0.015012302067978807,-0.02339511451080352,0.03973679631000221,0.029880164614531874,-0.002279413570361943,0.035235461123775756,0.004817302762095398,0.0015761152038679528],[-1.8657388357454452,-1.75894916125999,0.7185253564135495,1.1494304138630733,0.334936863414369,0.19923035142399348,-0.38473070793846553,0.029104310355363653,0.06406126543463499,-0.2627440491315979,-0.060443781517044395,-0.17769986474042612,-0.045216886633780186,0.4242614775279517,0.16075771612765888,0.09897456435327583,0.15990253305706453,0.0729529506898621,0.15985510472064396,-0.06414781224334104,-0.02891572203630819,-0.13853929792095773,0.004639596152724854,0.09722462860597414,-0.05096539532026567,0.19928581781298504,0.05105883197349335,-0.017649263961713926,0.1402821941083304,0.12602249743416524,0.12139952769119415,-0.11624752317635005,0.2061962979486954,0.01653191267500938,0.09277513250536384,0.1423345152398508,-0.1464424660220673,-0.0373798065592162,-0.12514730713077757,0.15637810452230597,-0.1532885945678566,-0.07622353768466021,-0.02751233383481633,0.09201275810431259,-0.02169766262959632,0.03639274564105623,0.01187552537510139,-0.06005402386209103,-0.07379065075175277],[0.026540820121259603,-0.44378471399071684,0.5216494478900174,-0.6655198624148834,0.26971378072419194,-0.3266210025841491,0.11820625501656626,0.1535327107322435,-0.13057039495309042,0.10581870028990638,-0.180770489505869,0.12996563025523508,-0.28202909326744585,-0.0444286333441789,-0.34190359834638184,0.1715355284057137,-0.45974197040884746,0.13080586988711407,-0.242562182429882,0.1598101089390879,-0.3931584633762105,0.05967221242468541,-0.2782338989793047,0.03925282375500408,-0.2995987261524855,0.04017893963080732,-0.3288200807189214,-0.007788515399018743,-0.34788052890843696,-0.040714196147577666,-0.4245833118770915,0.058217799420150124,-0.3748336766058017,-0.015692065764219894,-0.23537405149789,-0.07083055449136057,-0.14596354553590618,0.07616189760634114,-0.21496897850534336,-0.11246136254267153,-0.14430917609445695,-0.0985306545920902,-0.12803251990483808,-0.04729141946961863,-0.14727158693084097,-0.03915203867329782,-0.17908082297662925,-0.031917676472371585,1.58759986640501e-05],[-1.4804176195012968,1.309262368554988,-1.0579146866389944,-0.5878854032590874,-0.26360081232140414,-0.06782032312730354,-0.07980842021489196,-0.3049900100162899,-0.11183667440571021,0.17256752444310974,-0.10564781755755417,-0.22079403072379258,-0.18681685294373773,-0.12531655344211662,-0.11678208020395683,-0.3193232659711046,-0.178703170640052,0.0017540603338476021,-0.13921658215091876,-0.1386139186029945,-0.10897225513401897,-0.005139812214112867,-0.16243287453767385,-0.051397589536152255,-0.05166020306784437,-0.12230230306281725,-0.05504617561240752,0.0029136801587982706,0.09202649381908323,-0.03062967959690897,-0.024784169154569123,-0.04775883152163411,0.012147657270076835,-0.021558237173868167,-0.013782033113469141,0.06085640094652148,-0.0370268467684004,-0.1039112997656201,-0.01209958022097055,-0.12152176751184078,-0.017418211940353118,0.07600526546469538,-0.024993148038235036,0.04240823489841643,-0.008841497347442112,-0.01122568378167316,0.03532633380358082,-0.011017465963710255,-0.0011089348731060053],[1.345487970289883,-0.3927256915584474,3.178506764891383,0.08502204885644607,-0.14994768237182368,0.2246579376885908,0.1972639200972309,-0.2805583964139943,-0.028543881723234077,-0.14777212261805517,-0.07778020209469348,-0.3216649195738371,0.17412910718222677,0.02723162508237975,0.5924397214476459,0.19871693438065505,0.1977320654809868,0.2178008846151591,0.3814719667061806,0.15510091307983745,0.18960477056415365,-0.33261074538202123,0.11790314611146563,0.16273311233747434,0.22138162046842208,-0.12872581125498359,0.2178077293060661,-0.05785667277262304,0.27185509145795983,-0.1573832385444481,0.31791120777090454,-0.004522172373146192,0.2499193432313883,0.0037685779933165414,0.2558475747547773,7.397408971979516e-05,0.07427794041842753,0.24685487662442251,-0.042826160186854846,0.12356808620184727,0.14931111805267222,0.1033874252203188,0.15346081978011164,0.1832618927972841,0.25660646632836803,0.030040112658957382,0.08399270560266349,0.0703698068406767,0.11820588460682083],[-3.446184636493253,1.6498160052685515,-0.45278753074217093,0.1226042743807308,-1.051072362238173,-0.18899615982765477,-0.8773289411555207,-0.16071333360269474,-0.6569148509798426,-0.033917016804802774,-0.46600401726599705,0.11594604306237907,-0.28152476240283353,-0.1409365406774461,-0.1693690127503972,-0.1310044388920882,-0.17353198803218348,-0.29993335151010486,-9.45182696628396e-05,0.10411202536742732,-0.13259772595723224,0.08848490093672358,0.08320803702425514,-0.15172793410752794,0.09198744227688394,-0.02095179877696853,0.11922978461204083,-0.09055537598531595,0.04826829419639643,-0.054260198772814565,0.03749026858566871,0.0673490934822146,0.11501206134256765,-0.018653441721152696,0.06167360776033754,-0.07898968109065309,0.09178985267383323,-0.174497613445528,0.13755176877169595,-0.1401694566608004,0.08266844263227965,-0.06958129485863031,0.12186095699581521,-0.10120347942825221,-0.06001469678876766,-0.015235451150985168,0.12423956859340857,0.009630589681882984,0.019065318650954705],[-6.131357449124658,1.7461394000370654,0.4754342731472236,-0.4436807560995884,-0.3930967526721338,0.10342926004582291,-0.3464809873733819,0.048778011328658194,-0.023043865486430715,0.2192960185968083,-0.004050853667106264,-0.145763028592275,-0.3945223049540762,-0.1662303320074953,-0.06909470278166446,0.04205884273296918,-0.2045467869573026,-0.03696479011963359,-0.22148657102426147,-0.07510756960067641,-0.3062251085829175,0.03533332466065158,-0.14741949572215904,-0.15600813499431412,-0.1774737013209765,-0.011996060520605048,-0.07421886622351832,-0.045450975499774456,-0.22661875737283493,0.02412493898645351,-0.16733952892106546,-0.002726003164937409,-0.2001853599179243,-0.105384757768733,-0.05965126119530642,0.042654897293407316,-0.18397673584207258,-0.03909294895055429,-0.27822081994184017,-0.069187613428062,-0.14636223148534477,-0.035599374613199404,-0.09316314264038185,-0.049757826024224294,-0.10285646571884338,-0.018550238706126623,-0.0663586941221762,-0.011126684652864672,0.02702848478088883],[1.1914010059388103,-0.06592316816387035,0.6345049104512778,0.2566120320646557,-0.4199443347773477,-0.015541792809403494,-0.10578038949603623,0.040162268682204294,0.019930933312976455,0.05523708178821834,-0.03217279821163449,0.0016554632428002548,-0.011196751727766798,0.015028394363661213,-0.08986893000536098,0.03087406533678971,-0.011404931225344922,0.012214772229807927,-0.08071387567404804,0.004689381874669616,0.007989273054347417,0.023899680827982935,0.018097182852238734,0.007665224257602977,0.005299853182090608,0.01604529332880514,-0.057866495989367744,0.007777303070299616,0.03311288531498844,0.012998259345383362,-0.05589248851428906,-0.024919550839938215,0.00656436882219236,0.021935934931917304,-0.03440284119032666,-0.002389034381994188,-0.01103082425995844,-0.012885975582800687,0.017086272390017754,0.02530712023004316,0.024905395496782383,0.020747789105798532,-0.007898081871907305,-0.01902856180479968,-0.00873256195873902,0.02682002440537937,-0.0027464103483096753,0.003567194841385047,-0.03452631701852505],[5.237865462496324,1.7886187526740538,0.33235760443433754,-1.1810712958098453,2.234823849596341,1.4521114196261085,0.8205366604984483,0.7427375196753936,0.5331586312715266,1.0711677740204557,1.0799996496235493,0.2684303071695799,-0.3008814155884772,0.5717549596253171,0.30934279594099817,-0.018718577398034068,0.04450927518400069,0.9017397925434959,0.16363496553796658,-0.25583448683474497,-0.3097571420768005,0.4671518600703768,0.11313118523845471,0.5136122322695714,0.23161369484949434,0.04489219179638605,-0.27417434663075135,0.25927881511884326,0.32966388511756756,0.20434415686242835,0.31785952131941203,0.6927710449451107,-0.2719848931742341,0.2059913589851686,-0.15867704629767077,0.17291035582337255,-0.5740102211284339,-0.35791761136328354,-0.36891016481574085,-0.02665704297328224,-0.00686252382281371,-0.36069841100479116,-0.18378024766403706,0.21368175369660394,-0.22263142086878043,-0.06295975424337777,0.17111714045329346,-0.30631216316328863,-0.04137636256868087],[3.8804373270057195,0.578490339323734,-0.5715985927105937,-0.15632180998516904,-0.3259484389365269,-0.0011674632720832959,-0.24117938039455614,0.08441084627843544,-0.0575600546707733,0.06672372843070778,0.02549230501589025,0.014986571206399818,-0.051215571244172774,0.01689098144852458,0.025864952022539352,0.10490458661499585,-0.007108164151945403,0.025932075987291393,0.01553849085280808,0.03560536892427346,-0.06614952294910109,0.026983016266245587,0.003209861452407768,0.011562481644425129,-0.016863968137312932,0.00684873093097925,0.021292745411107428,-0.0030039870983050608,-0.053903612347074276,0.023352635780892632,-0.02832688183606315,0.07135709716853848,-0.0406671402048396,0.04437100576053886,0.012937964523152344,0.005452634288351914,-0.018786565875704934,0.009753744724072987,-0.014793145017442489,0.010056223182665129,-0.00719008250067293,-0.04116136186702636,0.0018625843770946338,-0.009760056867991397,-0.01863519340748186,0.004683175907379527,-0.014455006248736661,-0.002548983543317534,0.02119504555574379],[2.474178506498822,0.23480307702186837,-0.76766828238492,0.18686554211905942,-0.1428695985489606,-0.066023301860036,-0.04610966144267923,-0.031204220767140835,0.021165394955373142,-0.1021123955077262,0.036202091604789856,-0.053325612643430956,0.0964411707182813,-0.004987855124182821,0.0460278336342934,-0.03147608206413773,0.040318098951814596,-0.07153623829503834,0.054121291704461706,-0.01948139136452765,0.05676996022364549,-0.00843724894640244,0.02227137692862735,-0.008715823503540996,-0.019358270732549324,0.020406951693620813,0.002076248559121112,-0.01711746373159503,0.0038228837052792517,-0.021821916521727276,0.0063348477974181,-0.024591366737031568,0.0049806666357176885,0.006696582285851853,-0.0038117612899485963,-0.011129149335942064,0.023600882024705425,0.001657029737540064,0.03864300344220744,0.006795726019398213,0.0022568777952531407,0.011318550783145268,0.00017562106643227533,-0.012115879351180796,0.009793500010038562,0.005250717853645683,-0.00678163500288901,0.006427583570272241,-0.01040237201939214],[-1.9693198228949482,1.761483242077599,-1.9727923618864465,-1.5251796208298547,-0.8328103428918319,-0.20820586499926635,-0.5784367491872292,-0.2226067507120661,-0.01391151665492176,0.28132268364625995,0.1378383625205723,-0.3828556632683681,-0.4429876415736648,-0.07601162147597751,0.22042313292916071,-0.05725774340091414,0.13757438137841554,0.16915479676824266,-0.0447584919546604,-0.24049739702300243,-0.18079836404927263,0.01446710458652266,-0.008913123022702743,-0.0219931514806209,-0.02109256193483442,-0.14191807185428126,0.26352449174083364,0.08044240755455635,0.013247203642785206,0.15033631052409227,0.15800366106374678,0.18910406829903406,-0.08090147088294787,0.06607851440093032,0.21393473030091684,0.19891973826718493,-0.09442482660877087,-0.06846208056700419,-0.11328983956561682,0.07447237407756457,0.012120046601947555,-0.09273846475502931,-0.04155024241271668,0.06392053560781202,0.04513743267309004,0.046603002772995616,0.07035002291018805,-0.037308488302975854,0.0290212301818389],[4.78853113185421,-0.6645269480072219,-1.388800939106535,0.2693819665958148,-0.2577074167990014,-0.08597714556206322,-0.13332780127121416,-0.1443791523182836,-0.21179964480344596,-0.07873614731569895,-0.20547625177380924,-0.030257514906920576,-0.03530090534035371,-0.04249203345618906,-0.13229216861828383,-0.10065427952271556,-0.06190026371314858,-0.04367699778796049,-0.0984985717800306,-0.052866280566558135,-0.004984406269797202,-0.06097167662615569,-0.082702392120692,-0.0662270531353891,-0.014560853639468433,0.005218828154576082,-0.057894779257385294,-0.009711187633601421,-0.05209571928553542,-0.011292936085098024,-0.040473022844672385,-0.062198014531708756,0.004055936352737056,-0.04180119375565103,-0.02930741175606352,-0.022273528370546615,-0.02558534542519086,0.004238867678451312,-0.010044196468925917,-0.015191525868530936,-0.009923119998621896,0.028594823792964533,0.003825488905857921,-0.0317762856786086,-0.02113738821072516,-0.008480517047457914,-0.011045619413722308,-0.004953885651608577,-0.016022144463626493],[-1.6326316717155345,-0.03577533267789769,0.8432506265420117,-0.09694673237794445,0.42585165554703863,-0.0016208921244293814,0.36835077895598295,-0.07637939285411001,0.23986842043812023,0.040798958801486485,0.1308229101303599,-0.12738646010827126,-0.03458448092175204,0.0506176030513565,-0.000462940283788816,-0.11586379067738793,-0.13754097888950634,0.07076273213219823,-0.17554778225490708,-0.10294914374185232,-0.1110424041719117,0.04727635795531251,-0.15396011514999033,8.722618184072799e-05,-0.14262706197385022,0.016660949091303768,-0.1682812203747187,0.006832620997297346,-0.05872184876665497,-0.008226728645793883,-0.13355125748269467,-0.05629145330456962,-0.06779507997181536,-0.011523785014957379,-0.14915442341560164,0.018881466791233176,-0.157065267859625,-0.0033948871638415264,-0.1448361369309939,-0.00747649656734936,-0.11096895032122966,0.04544483642862676,-0.09588432416005335,0.02369326374057391,-0.10117241542054195,0.002564699399143077,-0.08423080941653611,0.008331205691478435,-0.10031811905328715],[22.03561207752719,-0.47793212002230656,-5.3549100576140365,4.328429641958653,5.612587679660662,1.9133439683314883,2.016848586249433,1.6292380959846362,2.6442000428939214,0.05593476544234833,2.198105676074216,1.2065358204567553,3.013012959570358,0.8664936152993613,0.6513389989884096,0.9667225913737788,1.972597075450805,-0.2849009762946612,-0.19516700074986246,-0.0004576294949065873,1.4526701627242076,1.2614757845663978,0.44767917311422506,-0.3374407813156142,-0.4081616758971174,1.243528610968975,-0.40159808167885874,0.27424110106005184,-0.7077397680784011,0.40013567773931646,-0.6809478028510083,0.19746751376667016,-0.2570357382735397,0.12486704621705506,-1.2273939760236978,-0.3506377463254599,-0.3416119054909347,-0.302688442625968,0.17635826292676307,0.024744488905301183,-0.32851467346895286,-0.052032589850671655,-0.0018208851234730386,-0.7187083507919145,-0.994528639539076,0.06419488072862833,-0.5796442999165647,-0.024657392939224813,-0.5215243515331968],[1.439045062123893,-0.20456221605606342,-0.7874869979547233,-0.041916478271300905,-0.022724962851745308,-0.05394106165932378,-0.20333300975840102,-0.04373604335458625,-0.17410835389533855,-0.08113956093283843,-0.18592589405881493,0.010173637704254694,-0.1108727427996152,-0.05495075425579741,-0.07547676583666123,-0.059425693505764564,-0.07963755436068144,-0.028867689715841003,-0.03381591229703628,-0.017343705791404648,-0.07187933331654457,-0.04674292669256434,-0.04991284874420571,-0.05175335652991121,-0.006405796970355101,-0.019329628207752716,-0.012636961413169416,0.006756476525094958,-0.031716410288101765,0.02899479972391289,-0.0020722702934417012,-0.0047247050216998,-0.011969363990384894,-0.028317882647938136,-0.006393626904030025,-0.00817302406118855,0.017977627321958112,-0.03276516424300074,0.01463845446567152,-0.027826023291929222,-0.031701035021157335,-0.010229159003611718,-0.003994368373360815,-0.003796982583025256,-0.0005813940867465325,-0.015150250960280115,0.020544565963751017,-0.005168910580237938,-0.004419719968710284],[2.407900023922214,0.026944679241877094,-0.48256953007517467,0.04191556555008884,-0.0628495382170967,-0.06136468354580547,0.022774290637170756,-0.007104923381445344,0.005208286318180004,-0.03233215455953612,-0.01592688442160227,0.005236231135945863,0.029789715397995315,-0.009418109921888957,-0.007472448785301258,-0.01741323250326827,0.01862874513933447,-0.04263700920724848,-0.008962580027482953,0.009935151438721214,0.006687114964430956,0.02052310603539761,0.007157667321328463,-0.0237402571127584,-0.021781980435227177,0.006135026539865572,0.009390527366630673,-0.005688961792566692,-0.014054043221290061,-0.018313548489330953,-0.009714968580813438,-0.020063939791395223,-0.00018367825525664934,-0.012072268957572716,-0.01165482856025234,-0.004738982363312996,0.0191385816239082,0.0002629735527996443,0.01714741853157175,-0.01770364433788624,-0.0023272142064702304,-0.0005078475389105997,0.010993273518105313,-0.01664351016432609,-0.0026963649375085583,-0.0005944525505095883,-0.009420858448125026,0.012623626713736574,0.0016819364747220717],[-1.3832365984549466,-2.541701185158599,1.4110011309465222,2.346687212018857,1.0794125932516405,0.4339580340649549,0.5429806659226065,0.48086190081339025,0.3447036953393551,-0.4770385272135712,0.0205676952947649,0.28299327948904396,0.10707741920443245,0.11838617309951596,-0.2416400555037321,0.11708266012022313,0.06463720201444025,-0.08668186816829494,-0.17242132138238128,-0.14888323029256875,0.21846016140290528,-0.013413765706397612,0.0028023127514994603,-0.015736233221577026,-0.17762172224787676,0.17089003214082454,-0.140436520106165,-0.038583521358269965,-0.22975788474706452,-0.05246678444439471,-0.1745350962251883,-0.2003088081559605,0.08350086305951131,-0.2052674877951643,-0.1703740743284357,-0.09710935239531979,-0.01901517655457814,-0.01164142312319427,-0.11163130432685195,0.0853567567181382,-0.06236978976482436,-0.07111408666638755,-0.04482573090359559,-0.13054047082452247,-0.008921853035312515,0.016708566879066203,-0.08933282564670186,-0.04998607131199858,-0.11761040007842835],[-1.18846415287388,0.01604440685095705,1.2679289950992594,-0.019745204634844533,0.22341168233253444,0.024001073291661252,0.30817853227136266,-0.024383603077788682,0.1824454011030416,0.086917320332545,0.17706372214028104,0.05624499087968431,0.18927060015791336,-0.014894436298902823,0.20886022258221104,0.018789359975989687,0.1667985665222709,0.06526528609790563,0.20253773651872262,0.03173097799810437,0.15180466332056286,-0.014642272692007604,0.15212166194483331,0.04273149458859455,0.20340921091771003,-0.0054000802098785015,0.14114883277918966,0.03397665375948266,0.15582696501198107,0.03727932779362766,0.1582622172270218,0.10018729040612094,0.126353230254698,0.05905196691787117,0.09347318408718268,0.021100637497310542,0.09592665664205605,0.050790718628055634,0.09256023392048228,0.044696260816761604,0.06490834462680804,0.04281610184404065,0.05055065607100071,0.05897058882807635,0.05470075793338677,0.034831189588826535,0.03401531346336731,0.0413290875692724,0.0024158206119947573],[12.008708029148332,-4.238906760214561,-1.4337212696564174,0.8791535761394775,0.009492900990388556,-0.0004771086738540179,0.060237593686365366,0.5283477661842017,0.29824239062225794,-0.3377415066869973,0.2699216551937955,0.29015523509770413,-0.03135594157748687,0.24578235549084887,0.020344670445940505,0.7059577488301652,0.22931214000968783,0.1303569870697272,0.09615190057463051,0.022818517193733363,0.27454970899603726,0.04504267507915242,0.3213822931291113,0.12163325798850756,0.05873491909813174,0.0841662244741685,0.25998530987419016,-0.04935870126202665,-0.27037100660911195,0.032031540266177806,-0.0700493994139409,0.015713466637554924,-0.02568418046994956,-0.09297723828557262,0.1847382852879639,-0.07826307139055219,0.09312404528088629,0.32468834702275756,-0.20366661767319114,0.2559388495567497,0.1456901951251485,-0.25892410625964946,0.09312133006862044,-0.07372175242170895,0.007277742149617964,0.11593722879169224,-0.20318412394504623,-0.003087117605518897,0.06300073870270886],[6.696975705045465,-2.18608060812487,-0.1622957433257488,0.671729283743333,0.1448406153611839,0.145914571464776,0.10514292623925806,0.28638806514447984,0.12568146963710516,-0.2014994796459422,0.14132446748929628,0.2652766128137853,-0.2659070352054076,-0.018442854859771284,-0.08446516786997461,0.35394550793204793,-0.029779023433665584,-0.02823406055929836,-0.04783852685453399,-0.003866267541535945,0.11375301574515975,-0.0547825259884284,0.10399126205860096,0.06536922998697543,0.07289121301771136,0.012693964740294255,0.049693350204765334,-0.02324740076584423,-0.18118556185188783,0.03850553450646405,-0.05836299350429654,-0.08922698298502653,-0.05225633228175632,-0.2333334284136548,0.10460136866764781,-0.16223619789868396,0.11229804448530215,0.11196123855659736,-0.03649081206560346,0.09191878767484296,0.08595535084484085,-0.1557434097778801,0.08223947966437818,-0.06548223693374271,0.05516901467812209,0.026857897099533453,-0.10008943738903348,-0.012689326275218933,0.046466442483512164],[-6.320266841974709,1.7651967404316586,-0.09131853986650032,0.3156955629099624,0.15326693459105428,0.2493126641414663,0.4389931075211496,0.43757192045247734,-0.19084025531770968,0.027915988780120495,-0.18271438731189063,0.37291343004678085,0.007672846504421342,-0.05692629141782069,0.09220489308799591,0.25924205946760237,0.28666181403241225,0.12687391340976076,0.13325979653433892,0.2994458695359788,0.12059033162886668,0.023212657443690658,0.08647477778323956,0.012118483507648497,0.18502154330829526,-0.02002347962067411,0.2636333312917763,0.12277783445626408,0.00029296181111600406,0.016421070575327747,0.14458255086055014,0.09646625502205101,-0.03321430282864542,0.022500798416140277,0.08418968460746411,-0.03889353217837831,0.12228302544185046,0.10098274744260204,0.17782404530808626,-0.024378381459348418,0.1383850717562847,-0.051670469843807486,0.11206653543830176,0.010158065692750845,0.08512320261494223,0.039998996635154195,0.07185066355460952,0.04704274651878553,0.14382144344173153],[-2.6048865807459567,0.35140733579885847,0.6853515326597067,-0.3211433283673304,0.0004103283738348937,-0.07487435274359008,0.13407492934948503,-0.06506920690499164,0.051304743820734286,-0.0028689755039630607,0.0021430644675456425,-0.07826724995800857,-0.07073125064293857,-0.0552469275220014,0.01917745094989923,0.030064535640855546,0.025063781202114867,-0.016591882611299704,0.01899378736614694,0.003085715961067661,0.015922800783080445,0.02627507594751413,0.03381352107549487,0.02022252630021591,0.040681374847956434,-0.04347944615339337,0.07927137145856158,0.014576448197206588,0.04057733423986142,0.006705956178532208,0.027724280888849824,0.03255991425641212,0.018149525730305493,0.005369018645768541,0.057763538406904694,0.03288009859021142,0.04954806141961904,0.019398937028599197,0.034153060528199476,0.010976164800710928,0.06228702487668532,0.00014656248068086144,0.03030792954383503,0.01923891511183566,0.03569667241747237,0.021800287202113335,0.011611036579366862,0.017723720754747884,0.023073848958472105],[5.565274644363271,-0.10359036801092608,-0.4738026510931778,-0.23777091640899192,0.047212833466604626,0.11304256477518787,0.10884258308286855,-0.09922341762555137,0.29154625599348816,0.2139643438209743,0.3978982491091305,-0.026120122015398803,0.017084893547912892,-0.012197848213973114,0.18461376794619613,-0.15110971463630737,-0.02723958967771371,-0.006086606582918712,0.12138283832123845,-0.13653452471556488,0.00737471386203199,0.15235902348120117,0.007652119525825127,-0.013878556119036212,-0.06708696605227886,0.17297119568241145,-0.048536795547522577,-0.05094624177045709,0.0539625149287418,-0.08821009145430359,0.05041831657927453,0.04577177972751174,0.075514957846337,0.04827284971833183,0.013578127530906967,0.0735294939826526,-0.12836001771487685,-0.055207787785797685,-0.11583331948693534,-0.016145258791776564,-0.0624289185390818,-0.014536024332678784,-0.05035533398823357,-0.00455609652366473,-0.04863070077180142,-0.059716846286202434,0.025525712837333576,-0.020241791891335834,-0.012833363679949725],[-1.4536467863882778,-3.227232594867418,-5.251390539854506,0.5521218028968505,0.8286318668525331,-0.021929926382517104,-0.46836456178843805,-0.17805278468921065,-0.10042435074410123,0.12525797171248482,-0.25103095558643607,0.18174755449451,-0.22193440037159812,0.2019676867502518,-0.3989013737110938,0.08248809959926601,0.018035837334984997,0.3165733254890936,-0.5042310919434864,-0.2811227505827185,0.0879795004810573,0.06198536720660855,0.09187752443022094,0.000900172682377843,0.0816483627186707,-0.0698790013165237,0.008686756338652183,0.16892457078003154,-0.017597612564071767,0.1081226725920935,-0.1214870504347117,-0.11318949313731436,-0.2628033463801783,-0.08285701324886871,0.003521245077754903,-0.11193921794023583,0.005731230696067273,-0.0016612186766399796,0.06558708840538019,0.10583730993460819,0.12048741848964765,-0.21891329755557723,0.06564622891351776,-0.10628827053750915,-0.10559044376940423,0.05573216216216908,-0.07793808375128725,-0.12200152082086341,-0.009583708858972235],[5.563211289888178,0.25862339732220274,-3.176604999544622,0.15057027969688322,-0.0817068330997971,-0.2502327759067173,-0.2500868720218168,0.0946206950210671,-0.3103162705847721,-0.23436018921181043,-0.396443772966687,0.3336286200310825,-0.10768649779737823,-0.11920690553106056,-0.11910269104324861,0.10188823993288794,0.12262308276218661,-0.13872860017312566,0.16065159048645297,0.1747313591485993,-0.005349376410069517,-0.042115306214298764,0.0739736211402189,-0.06074900926529198,0.003556592096032054,0.04289735650941161,0.12371847370717234,0.012027060064973063,-0.08025685834754323,0.007764401826154257,0.08313057199324052,0.02215841150910237,-0.024743105517712922,-0.004091583756260837,0.047054536083678516,-0.05821077356527275,0.15491204331701078,0.022154770704288895,0.1794200819500065,-0.041943142413532745,0.019512969898003864,-0.08751335014941032,0.07993264717691052,-0.05412023545079415,0.10976284965146546,0.021545109942227185,0.06882458170193928,0.020657054131964316,0.07212985366560383],[0.12170426881745508,0.37216986854169143,-0.2598494609951641,0.18902357357628208,-0.18188138266684478,0.06933526669818206,-0.06822283701903845,0.050194516015622095,-0.03507558724361367,0.015371710978237562,0.01878731366340971,0.03768365147063808,0.045930522066380035,0.01546936621826922,0.0695897593284868,-0.015864746748897884,0.08834342648282681,0.003339495342475094,0.0694807773676091,0.015098863729476846,0.06513566651428518,-0.0028631646320412193,0.0465885639198241,0.011168733257491268,0.0685572660652778,0.017292834712868594,0.07611822026308199,0.01605197466352894,0.04501326316677466,0.017568863574605836,0.08193593828234833,0.03391235241871789,0.03909988684087974,0.02585089548502248,0.030593116516029924,0.015448788764861696,0.031460193021877865,0.009550952594576799,0.03279630668930862,0.029958226113711624,0.008476575350099802,0.0020137364601572776,0.01772242009393506,0.024159750076311312,0.012604837880703402,0.02203194939920617,0.007615594210662158,0.0056113489548043115,0.0017623176489572462],[12.09787049955581,-0.0124622990152875,-2.614734025379969,-0.5912196348939173,-0.4249729822651856,-0.13224475934066846,-0.2969329671526361,-0.04771305417564205,-0.05755000014761562,0.1074395437555862,0.049921199961002784,0.0009953361010431785,-0.03342583974338119,-0.13196745952205538,-0.11876377065096201,0.00689729412708427,-0.10195505982697659,-0.0037298650748583775,-0.10557018318438272,-0.008959307921875099,-0.027955072337275504,0.04949962626332015,0.014726305024882242,-0.006973719730972981,-0.09304229564291298,-0.054121908118274754,-0.08964725492851575,-0.02009852687792025,-0.15023829915139952,-0.05596974981082827,-0.10772686175847081,0.03856517810490383,-0.130132896879251,-0.04478451719170765,-0.038238278167684096,-0.05049386248069257,-0.02358739434825538,0.02506234905121445,-0.06644613424965948,-0.04386393789490061,0.04404869233617445,0.0007716365960621403,-0.04114606005128989,-0.05322233276416191,-0.010097464861088076,0.018010171500638293,-0.038003751996379544,-0.01418589776480425,0.039892096655649883],[-1.6835073395192794,0.34696353356158177,0.5293778934713123,-0.24551976608990703,-0.20043685978445513,-0.05379179110355037,-0.17806284686588783,0.16007273057885957,-0.10458968633085473,-0.015804480388805216,-0.0614837625334545,-0.01077015338428819,-0.04707658673761645,0.0251488301919312,-0.0840391645097841,0.03574065358517794,-0.10633414192639092,-0.059679465078661856,-0.026822602435800397,0.04093938645926223,-0.1054748133440315,-0.0038878446196368515,-0.06317047280610671,0.002620816920593744,-0.08027495240548632,0.01086850647219767,-0.09157845835683052,-0.024336503346630445,-0.10508849962841023,0.03210182004218799,-0.0594469598480207,-0.027879742146163865,-0.054395024970734875,-0.01393942676749899,-0.07882554739991678,0.0025568480435170563,-0.013758753199542654,-0.039926289032570374,-0.053658097303170435,-0.017128290559723047,-0.05821977534450468,-0.00521546990269979,-0.035046067132665906,-0.023335606203826846,-0.020564203298891918,-0.037639046604205646,-0.030048359387596636,-0.0059836085763366005,-0.006798409185286049],[0.03351204339102368,-0.21377592555259484,-0.7655366012634822,0.1776700993900552,-0.3528563536295595,0.07168477243146394,-0.16227865756619078,0.12507375452983682,-0.20964654604730099,0.017009303908511814,-0.0627423454079215,0.16720543084689027,0.11557771713120923,-0.015413513472879833,0.1568200496436492,0.23547175249193694,0.28402502262788903,0.07440630624044102,0.3087575616747997,0.24858462332439316,0.2169589656985519,-0.02636531826069552,0.21761321219096053,0.08050690440857307,0.27658493037965615,0.0513638570576019,0.3064190482028264,0.08501545733253528,0.10821146429631515,0.08628748687787739,0.21191083412403441,0.21610609172560877,0.06242517143816424,0.12500865180861834,0.2033026188312311,0.08465202296748493,0.18240540278108833,0.15178445771567742,0.15290187578781606,0.11472298845662185,0.14006469318837333,0.003898506255675137,0.06595404340903305,0.0706368514933588,0.091665172676456,0.07243735275336548,0.04604506482638785,0.029653032362394773,0.08472609803573317],[25.504555625032545,-1.4828712811501943,-4.141849300015865,-3.114387425328459,-2.068489010995578,-0.5007906482197332,-1.2309342365491849,-0.01549658404144061,-0.04475145646544652,0.4496618391723015,0.4945165816455311,0.25124612406733454,-0.10947207720312715,-0.19357334140901197,0.9059303096441549,0.4607608759259052,0.16301818463261208,0.9837974137570299,0.8555067775911916,0.638104784328211,-0.3520301607966178,-0.267985850114126,0.6387193314446553,0.5508937022466397,0.6560131575908275,-0.5363765221498791,0.677054307858969,0.2493878760825279,0.45562199694362887,0.21136557013766727,0.7626973167602359,0.6180213344083544,-0.01803255150874209,0.2948290226360441,0.9565874219979484,0.37195383670797255,0.2742419018858322,0.3308162728229787,-0.050969026878367484,0.18191895651566461,0.26105926710895916,-0.12070732505474,-0.20994045531870412,0.4879251783282646,0.48324925745471736,0.03332917887053406,0.2108041396690682,0.02078991547782075,0.2807671810097211],[1.1304843326021303,-0.2681189097149764,0.051183300940971246,-0.08222589223773638,0.1899853894688607,-0.005774826732269276,-0.012973245476044724,-0.02594741184593237,0.03397395319934005,-0.02647074387596106,0.012191083755587029,-0.03843825455976345,0.038515035488421966,0.010636203030091388,0.003905292704447841,-0.0025257989381361036,-0.02482093811664528,-0.022602541497755486,-0.01723225115065125,-0.02719818790542336,-0.011230963320787418,-0.0098156753313732,-0.03812709980404891,-0.029898200831367128,-0.045459089594273326,0.005054309559352296,-0.05704377438196317,-0.019901477437555985,-0.02954224486619258,-0.004082131576531411,-0.03473328330665539,-0.021766427812470516,-0.010283943176688954,-0.004674420537409126,-0.03687180937530798,-0.02056058599223532,-0.00753742585200243,-0.006957107119704741,-0.023636769740607296,-0.018334407257642755,-0.037524428890194826,-0.0023019064103213522,-0.0012809533506198062,-0.004874813086701356,-0.0193524230902899,-0.016450035992774663,-0.02382263477669564,-0.0037957869637100057,0.0007923361305776335],[2.347268293529879,-0.2731531589681078,-0.5099669443886388,0.11831170598669244,0.09496821420862768,0.012874722854765378,-0.03723968204791464,0.032188857464866584,-0.008461731606876839,-0.044983540919804915,0.0003592558102670473,0.03489992910677302,0.03291774842085023,0.003398939825972201,0.0038194752446311423,0.0006167090069848746,0.016401794794865292,-0.00547931615703587,0.008614679506453872,0.016806951045209132,0.0063907222053132175,0.0038362182738007904,0.01809469347889117,0.0038298006318328373,4.8856919405960576e-06,0.0165805795769966,-0.0017486953762584864,0.009709151389019233,-0.009566724940860612,0.007095332823062859,-0.002510435240583896,0.010152666285111236,-0.005965096758253638,0.0007070299209230403,-0.0003672261198304901,-0.004015044530310691,0.006760519240931069,-0.005748255128621787,0.027016579479517227,0.0013021136272481023,0.0018932138462103535,-0.004331210860636169,-0.008142664428018387,-0.009369871341850529,-0.00431483684484331,-0.0012147103469072322,0.0012889163443208047,-0.005919920055515712,-0.0010677579773803124],[5.694843502144458,-2.759678047812095,-0.3467861666377773,-0.6784992501974735,0.3935866810666933,-0.4048653504233719,-0.018470349570816603,-0.5012929869973415,0.3235254993611101,-0.06821074082749269,0.21358480112135236,-0.3977611946938985,0.07191672334125779,0.04803286115959583,-0.06686634801867114,-0.14787371456806636,-0.24410948581576444,-0.06984392096026924,-0.14801048588082094,-0.29896134850957073,-0.025552513972079718,-0.019204604152016424,-0.13824247374304402,-0.07791699520323452,-0.189320742732792,0.000897733944520453,-0.21228247164810632,-0.11974661853479267,-0.06904308964760017,-0.06531309870029993,-0.21874376602110174,-0.1265815778443129,-0.04482826882821349,-0.0009803381725283767,-0.10190991019426286,0.00974944825672943,-0.0735797728744616,0.03226218979429065,-0.21452754645690522,-0.012625797133772702,-0.1378166726717238,0.04526070041148352,-0.09272760805294931,0.01245802146918623,-0.11674339517457902,0.018009161837410883,-0.1673888359672933,-0.02673771199455075,-0.05780961481237178],[2.0729904491054763,0.6724593434480348,-0.014508420441653225,0.17106863781122641,0.7765899527665502,0.28156771251230805,0.40738695527659957,0.30865094572967905,0.45804301128834873,0.16098852454130025,0.4171023847537805,0.1181949886412027,0.3134749009038091,0.2099329092497282,0.10107879031656544,0.1708004168998608,0.13246085618261041,0.023236164238206666,-0.07427096897247078,0.0031519698606264344,0.0023999158382305044,0.17859458713430676,-0.08781251549687377,-0.019063714061370897,-0.25280860408443306,0.09211125761403782,-0.21067278031738773,-0.009519138367028434,-0.23324348566848713,0.020958505186480174,-0.2271149434812793,-0.04440405466066031,-0.17686671279953292,-0.012330550993962835,-0.253736377121569,-0.04704809683568217,-0.18274281112620508,-0.08483756665502375,-0.1477997805597557,-0.053102234738397314,-0.1670546426825206,-0.05156687719574397,-0.11218583837283472,-0.09522524982512406,-0.15072001779281793,-0.03158365827282863,-0.10665948580154039,-0.04470955013011286,-0.06396421604114813],[5.03630667498938,-0.5517824886638973,-3.2812448266703687,-0.46782962196196,0.4492290229715706,0.0125965306336471,-0.2949118031063646,0.131508090517382,-0.23782406676491735,-0.04893972037813608,-0.10452062503606209,0.11162840449526493,0.14442613671560026,0.09363984439327581,-0.03707336236921646,0.029505518427360825,0.1169882375097151,0.03939957138855181,-0.07800167644546,-0.05946375231245422,0.08999933524954731,-0.15870555481576956,-0.009703418401757693,0.05052322295850206,0.05010075484126499,-0.07053454720601951,0.042700225100850475,0.08621526186595009,-0.06230705825264512,0.06564626843491968,0.13550634335317777,-0.0071740457659005274,-0.1687702092852461,-0.0043737546501911966,0.04370842383400003,-0.032862557044239335,0.06075616408010365,-0.03229832134247129,0.10126529048432027,0.043299645752969754,0.014753152239197312,-0.05935127589132027,0.003980140277165174,-0.003880664263580872,0.08949761554027928,-0.05757252802573174,0.008589748967354214,-0.08975563024558246,0.05329056520693125],[-3.9871176040619,3.3131113925269293,7.141717577726507,-6.42050475899531,-5.401147194072951,-0.9121756338474579,0.6962328284907343,-1.6530502339103512,-1.1988759916858782,0.861493708587107,-0.7014201461883521,-0.921643253672781,-2.510198752168347,-1.5175126099408915,-0.49459603180806916,0.06177180955936192,-1.3297002936211773,-0.14998604293143092,0.2812591164753566,0.4085132937294146,-1.1104243487413596,-0.43416169163838975,-0.29236527220447006,0.28828589180054087,0.4130900798411408,-1.2030462757176397,0.19988395766823816,-0.20651213036455732,0.2978792339295259,-0.15540437866049406,0.045995746145162865,0.16453341737739696,0.16983765761013772,-0.3454805187111015,0.9826313982483817,0.35892908067274576,0.16557037342172856,0.19253883645273331,-0.2187752241037595,-0.15313400668620206,0.40368953455784085,0.17315559302276987,0.07575990621789559,0.3145133249668052,0.7622304825947094,-0.1603951054667959,0.5216466189428174,0.1748229789488251,0.36683948535447675],[-17.141226667068125,0.4514185682535702,6.644657779786167,-0.053136066358533754,0.03182667827004611,0.1099749107188025,0.30798062187577396,-0.49553321936856437,0.3995870071992704,0.5282749605208606,0.18391513375493304,-0.07017654486475727,0.2553823758421725,-0.06797560115383351,0.47383361500215615,-0.19103610470760893,-0.2630227325775647,0.5785341316537147,0.06627154449868204,-0.1916003212668806,-0.06716725328611238,-0.2005932392116996,0.02938026611287118,-0.0022832612559943277,0.3452668574936811,-0.18169226196385413,-0.11252591659353862,-0.07839882707037424,0.19780887509375833,0.019847846858525305,0.26091035739161994,0.06971277267734623,0.21358790819205387,0.07112369116281826,-0.004442273594451267,0.009466646250251166,-0.051305858611775175,0.13191073720874635,-0.26518663196705017,0.0038237497559608846,-0.12227417299093139,0.1732811063748483,-0.004109089946411988,0.20669262150141046,0.041067943377842774,-0.03010074884354725,-0.0832005293496659,0.09919218111280263,0.036640029584395044],[1.0685860805904448,-1.0092115460261413,-0.8481998460531537,-0.2994408973896727,0.6962175631196779,0.06512124961472027,0.18901338735500622,0.01390092998703048,0.15659434676218728,-0.15185640854966095,0.10328488833880264,-0.02898364268540414,0.059271228063932875,-0.009712810629395038,0.045334655351195685,-0.0018568501103090713,0.0261633772767726,0.045744328365990655,-0.01916531053312363,-0.09592171745878025,0.05741796020642223,-0.02898629939688385,-0.008093166492994268,0.02521693061959138,-0.011344805143043507,-0.06727455093447639,-0.009003218399615428,0.015950332677893538,-0.0018124092050315697,-0.0039409433697950354,0.017433710492182317,-0.06474544202584115,-0.03514152530106611,-0.036200216793341705,0.034306519943054996,0.022384010152152356,-0.006909934210877072,-0.005146367361943483,-0.00035779647904808697,-0.026593879257984645,0.005264380769034654,-0.014207273791627178,0.0005445307800388253,0.02262279485795601,0.002407559669196166,-0.041032971931633126,-0.01113004208124896,-0.02989518143119084,0.026194708083533184],[-15.045021696972174,-0.4409032762985972,3.4119786969854315,2.5869426677202174,1.4120426310634484,0.550808071991265,0.4848798561074243,0.3655594734777908,0.04976278057836933,-0.3607421259099627,-0.061022650844740234,0.5362781937561467,0.6273569859172125,0.1075730519837408,0.2054061339914228,0.027269064834847475,0.15001135829010637,-0.026114163410948032,0.11268188735774304,0.18640351173317568,0.15270601074661583,-0.17907440061104302,0.10874036374107997,0.05704905688925888,0.11771809942942321,0.08357378083647492,-0.11923692529760965,0.09185373253978167,-0.03134311712685885,-0.02468353335908476,0.10128999257500924,-0.052006565719558556,0.014469138200070378,-0.09908659321408711,-0.2158857045774843,-0.14117419535187933,-0.012147826713724746,-0.02510760425863418,0.11554343983059011,-0.03972778410756182,-0.13635294105773324,0.061446602098615355,-0.004354052009217456,-0.043681314204518436,0.0078058161050968645,-0.039841143708764896,0.012573362525530528,0.0004756101063346263,-0.04163412528756276],[-0.31077769968155317,0.6635949298517668,0.6242624104449386,-0.9971077977063153,-0.40802978655409494,0.07083091168609186,0.951040446186909,-0.4003456470018569,0.11161549343490275,0.3206208019456437,0.2509585862097123,0.014365992486103863,-0.34410385107928515,-0.2663313002621704,0.14234112743225596,0.03885824991193944,0.08709307355636467,-0.06939218884993453,0.29336407076960325,0.14784516740619036,-0.04951210172196622,0.114895418398384,-0.06928613611963907,0.08877309883058976,0.14039231678684408,-0.20811713810216653,0.21311712547031444,-0.0008906446978320959,0.21909478847257657,-0.11083387765822475,0.07165732278872397,0.1260343140729933,0.12590286847598203,-0.08709111040611901,0.28950107005029835,0.049380214627518036,0.07616830262124041,0.09180945296311611,0.057444511019815146,-0.02810392818958633,0.12684133157699468,-0.04866579736433472,0.0673590786461815,0.09410397759648355,0.14935187417021695,-0.023941842581510747,0.12400704763542963,0.0323055854859638,0.11830787552585054],[-32.79232284042135,-5.21309007545656,2.6433832665632675,-1.1119207661884076,-4.460106114932221,-0.85685873452816,-1.3653009389554747,-3.215611514444982,-1.0708799319697244,0.7469056417386362,-1.4621900573113793,-1.5865704523642161,-2.0681576536363395,-0.7666987962305579,-0.2828093627696048,-1.4381287360216382,-2.2980964250820755,0.508883913057748,-0.6417544941659329,-1.0514079325575223,-1.1487812879740822,-1.0042123957472397,-0.9709497623262457,-0.6326224212434383,-0.25304611957945516,-0.5135348620744662,-0.6198241905477874,-0.7439330605849016,0.31479873311957823,-0.46705390111135625,0.038488088259295214,-0.6779997882967208,0.6408565988639838,-0.25585495065500663,0.13200280547267287,0.3154903746260627,-0.7467475312366669,0.23144914662856494,-1.4631741336557786,-0.27268679719752903,-0.5960940085278037,0.4594091005677105,-0.06257547042002728,0.4622366532674076,-0.004391005595976451,-0.13123687712194512,-0.027116910044965352,0.12285669945288477,0.07952302853649744],[-23.649285790413074,0.3407898617714277,7.224594771786535,0.7273817855754507,0.658670366014461,0.17439968865602412,-0.10921528614152946,-0.3168145812453719,0.13283010782218876,-0.09563359355614905,-0.38504291143458835,-0.8699778277840214,-0.4169352537579982,0.3687979030925906,-0.4794666634009826,-0.5054395695487555,-0.3686467761415155,-0.16276037125094528,-0.6180971929499446,-0.7781335500003235,-0.24714008485580133,-0.16618714198109455,-0.781215932020588,-0.23400651656038574,-0.6883989839665393,0.17807535084240522,-0.7113190212941385,-0.29189887920916147,-0.24185802725904146,0.007015980925285618,-0.38547899784082074,-0.5264233246766284,0.17055401427818126,-0.16227253653910056,-0.5718515896264939,-0.0521496888721183,-0.4948401878366447,-0.3636036415416811,-0.4168795733246037,0.007031419725982875,-0.5253631201291671,0.017988765021447105,-0.09370643628827453,0.03706230340700279,-0.24636408409580432,0.033682861513375476,-0.15203161023227746,-0.11404617264995087,-0.17685138275072987],[4.076457104137737,0.08973136800772254,-1.0565529481956109,-0.301709024693608,-0.0841697381384705,-0.03545630661752948,-0.047257793935234334,-0.04233823242355513,-0.03610259972983205,0.05179225963510427,0.017677310720295943,0.05127775103995566,-0.10298340321911528,-0.06029858443115669,-0.07766819594833554,-0.023404976749153553,-0.019688677010443037,-0.029905372143614408,-0.024287893869475746,-0.01149406519528244,-0.02258274442749346,0.05839639351845947,-0.031675399498534706,-0.008773353798892485,-0.014803870037982796,0.011714222064343414,-0.011665026730194698,0.01351904577297104,-0.006423642266665354,0.012490183944800613,-0.03359690659934898,0.02564465386455643,-0.033401519892188744,-0.011540431972864671,-0.00386081780030666,0.0005696475963160525,0.006275475821173343,-0.03480091502644171,0.022854013742475407,-0.027147917550848805,-0.010534029718466523,-0.025644999419694007,-0.0005304974342087876,-0.011835159009082765,-0.020394084180789213,-0.003936586113469843,0.0013832915550914323,-0.008665757810588473,0.002154390992053234],[-1.099349308222994,-3.3792127996892387,0.9330344699591427,-1.2101893125922345,-1.4896142224088575,0.13454399199607422,-0.4948061635288385,-0.7917018560084964,-0.2639530668733761,0.26162671242022206,0.07002152554277101,-0.8259930927983743,-0.6235029327788707,-0.31543726300252983,0.22282349209003743,-0.37262458724476594,-0.6146119982559458,0.2514009572074771,0.0014310634933488156,-0.29684099859752,-0.21696360194932904,-0.24463658554226148,-0.3542654861141152,-0.0648890783228111,-0.045881520015427994,0.09332265627463453,-0.15030797793487993,-0.10447518032579375,0.12619413030981969,-0.18015636611467825,-0.0009381281340494824,0.07399122776643294,0.20408476578086188,0.10223570321186191,0.19708419855389825,0.2765651168347508,-0.3802389608867943,0.08222192948226277,-0.4926032824440331,-0.00666003091068672,-0.028135293409246195,0.1791550447728116,-0.11837072811170432,0.14273141031344494,-0.07480206205124751,-0.14611606060522883,0.07229282710521061,-0.027419610865356957,0.05475131101358933],[0.8572064728622035,-0.30757397915903845,-0.6248230249996727,-0.318479943175889,-0.16519445690058326,-0.16404798259758388,-0.10822526578876779,-0.12022971725371,-0.09642701064811363,-0.02454100119947713,-0.0848529756014374,-0.06294820832990033,-0.08729290612985079,-0.03509720196962141,-0.061574951016898215,-0.06165447487822818,-0.05841933863263874,-0.03172026137907828,-0.05133084005601852,-0.020575963637467977,-0.05462308306069489,-0.010374122159990506,-0.043832987796017166,-0.02421886086447115,-0.04260791117463919,-0.02329414251191148,-0.030840531782322954,-0.01214500705873115,-0.013837744926322814,-0.016621734335818348,-0.03081888312946126,-0.020669012522583324,-0.02730106547101388,-0.02310431200794611,-0.01582821296011776,-0.009918882995634381,-0.015748240784237606,-0.013198597729568546,-0.01301407654354679,-0.02190948973186059,-0.007940128580797012,-0.0016466885315670377,-0.013885649222693556,-0.005290229882482592,-0.009942427958918872,-0.005008271393644004,0.0008099992496559363,-0.007583741809302899,-0.0047056186638527895],[-7.933515217134772,-0.5064684793882688,1.8202518930752034,-0.3881893491577684,-0.3178736426283989,0.03002772662105259,-0.11045860179649615,-0.12195954966238083,-0.16725213362568653,0.1925478981041828,-0.2387161285401338,0.007478506785894132,-0.2782472475262904,-0.11905795672823025,-0.1148621768069501,0.03400578656897881,-0.17717997202750266,0.141018741377447,-0.15320436332594303,0.03412096562441786,-0.14023287442783947,-0.06277883720428905,-0.05014273830451042,-0.0113665760312409,0.10025533052091071,-0.08100920436351984,0.022206206919951694,0.02331259228882106,0.027363963379491017,0.02648369670558316,0.012541764348433477,0.047107123095342124,0.022515630088690793,-0.03537478380280154,0.053816549873153666,0.010276866274530887,0.0035242976870476584,0.05499804946951643,-0.09617221813062236,-0.01412366499770708,0.025000159450868555,0.021460228717155066,0.03403742602876299,0.04990415138455264,0.008942421556067396,0.001372108605559949,-0.009332287306772666,0.02194778167211731,0.05017770112467747],[-3.394645681324655,0.7428428572123137,-0.3517004634318889,-0.34104346477872904,-0.21348481003176772,0.09246255500121575,-0.3028998365269049,0.13740786750029377,-0.3437354503523097,0.20878462083289315,-0.19054490796835777,-0.053846573817691416,-0.1621063410492151,0.1185986198984139,-0.04039928049456648,-0.001735541491354646,-0.0032897921978053846,0.16117289495142753,0.012977551842485736,0.040111383087249965,-0.1496237030981004,-0.08842246454497418,-0.017855672733053538,0.07523995507483149,0.03178393348092737,-0.05959972110774226,-0.004860902275869381,0.03779223798765132,-0.02326663707073407,0.07514980725421536,0.13723837854475077,0.17248854169525174,-0.05511702225334809,0.05658853980199488,0.0217727191538528,0.06330888313996053,-0.09973579328215626,-0.055538810985482134,-0.04339502388649184,0.05694721543417052,0.02154993080895303,-0.03555467491457003,-0.04875150425095857,0.04881242297697554,0.04295988833203726,-0.013981592663696175,0.06946240367770397,-0.08172403524562527,0.03255661189081589]],"training":{"train_count":128,"validation_count":64,"train_seed":42017,"validation_seed":89173,"native_phases":129,"triangles":800,"primary_mass_solar":1.2,"primary_teff_k":6300.0,"passband":"Johnson:V","native_artifact_sha256":"eb4936db6d19f602c227b462bf2b0969d9b83d59dbad750a7af2f7ff2b85d2e9","baseline_commit":"40b350956d177be2fd078fd4c5ddd6735224214c","metrics":[{"regime":"all","n":64,"rms":0.00855089374435902,"curve_rms_p95":0.017097432527132514,"abs_p95":0.018668438731105852,"worst_abs":0.04077584332084805,"depth_p95":0.015250321878910537,"ingress_egress_p95_phase":0.0078125},{"regime":"shallow_contact","n":13,"rms":0.011231370122190864,"curve_rms_p95":0.02159871244095104,"abs_p95":0.034845420978071834,"worst_abs":0.04077584332084805,"depth_p95":0.028333806181268822,"ingress_egress_p95_phase":0.0035156249999999667},{"regime":"deep_contact","n":24,"rms":0.010481628292936378,"curve_rms_p95":0.023927179541075427,"abs_p95":0.025057624239186116,"worst_abs":0.03494305660580932,"depth_p95":0.01797756110620208,"ingress_egress_p95_phase":0.0078125},{"regime":"unequal_mass","n":21,"rms":0.011590169461699912,"curve_rms_p95":0.02499368858193462,"abs_p95":0.0315751748394885,"worst_abs":0.04077584332084805,"depth_p95":0.022621299439232623,"ingress_egress_p95_phase":0.006640624999999956}],"selected_validation_rms":0.00855089374435902}}''')

In [ ]:
def contact_preview(inclination=84.5, mass_ratio=.75, fillout_factor=.25,
                    temperature_ratio=.84, period=1.8, irradiation=False):
    s = CONTACT_MODEL
    values = dict(inclination=inclination, mass_ratio=mass_ratio,
                  fillout_factor=fillout_factor, temperature_ratio=temperature_ratio)
    if period != s["fixed"]["period"] or irradiation:
        raise ValueError("Preview requires P=1.8 d and irradiation off; use native_flux instead")
    x = []
    for key in s["keys"]:
        lo, hi = s["bounds"][key]
        value = float(values[key])
        if not lo <= value <= hi:
            raise ValueError(f"{key} is outside the validated preview range {lo}–{hi}")
        x.append(2*(value-lo)/(hi-lo)-1)
    x = np.asarray(x)
    centers = np.asarray(s["centers"])
    distance = np.sqrt(np.sum((centers-x)**2, axis=1))
    if distance.min() > s["nearest_scaled_distance_max"]:
        raise ValueError("This point is too far from a training system; use native_flux")
    weights = np.exp(-s["gamma"]*distance**2)
    coeff = weights @ np.asarray(s["weights"])
    phases = np.linspace(-.5, .5, s["phases"]+1)
    k = np.arange(s["harmonics"]+1)
    flux = np.cos(2*np.pi*np.outer(phases, k)) @ coeff
    return phases, flux/np.median(flux[:-1])

def contact_bundle(phases, triangles=800):
    b = phoebe.default_contact_binary()
    b.flip_constraint("mass@primary", solve_for="sma@binary")
    b.set_value("mass", component="primary", value=1.2)
    b.flip_constraint("pot@contact_envelope", solve_for="requiv@primary")
    b.flip_constraint("fillout_factor@contact_envelope", solve_for="pot@contact_envelope")
    b.add_dataset("lc", dataset="contact_lc", compute_phases=phases, passband="Johnson:V")
    b.set_value_all("ntriangles", value=triangles)
    return b

def native_flux(inclination=84.5, mass_ratio=.75, fillout_factor=.25,
                temperature_ratio=.84, period=1.8, irradiation=False,
                phases=None, triangles=800):
    phases = np.linspace(-.5, .5, 129) if phases is None else np.asarray(phases)
    b = contact_bundle(phases, triangles)
    b.set_value("period", component="binary", value=period)
    b.set_value("q", component="binary", value=mass_ratio)
    b.set_value("fillout_factor", component="contact_envelope", context="component", value=fillout_factor)
    b.set_value("teff", component="primary", value=6300.)
    b.set_value("teff", component="secondary", value=6300.*temperature_ratio)
    b.set_value("incl", component="binary", value=inclination)
    b.set_value("irrad_method", value="horvat" if irradiation else "none")
    b.run_compute(model="contact_model", overwrite=True, progressbar=False)
    flux = np.asarray(b.get_value("fluxes", dataset="contact_lc", model="contact_model"))
    return phases, flux/np.median(flux)


## Instant preview

This runs in the notebook kernel. Unlike the standalone app’s browser-local JavaScript panel, ipywidgets/Colab controls make a kernel round trip. The frozen model itself is local and does not call a service.

In [ ]:
p, f = contact_preview()
plt.plot(p, f, label="contact surrogate")
plt.xlabel("orbital phase"); plt.ylabel("median-normalized Johnson V flux"); plt.legend();


## Observations, residuals, and authoritative export

Provide phase and flux arrays below (or replace the example with `np.loadtxt`). Preview residuals are approximate. Export only after a native recompute, and keep the resulting parameter dictionary beside the CSV.

In [ ]:
# Example parser: CSV columns phase, flux[, sigma]
def read_observations(path):
    a = np.genfromtxt(path, delimiter=",", names=True)
    names = a.dtype.names
    if not names or len(names) < 2:
        raise ValueError("CSV needs named phase and flux columns")
    phase, flux = np.asarray(a[names[0]]), np.asarray(a[names[1]])
    if len(phase) < 8 or len(phase) != len(flux) or not np.isfinite(phase).all() or not np.isfinite(flux).all():
        raise ValueError("Need at least 8 finite phase/flux pairs")
    sigma = np.asarray(a[names[2]]) if len(names) > 2 else None
    if sigma is not None and (not np.isfinite(sigma).all() or np.any(sigma <= 0)):
        raise ValueError("Uncertainties must be finite and positive")
    return phase, flux/np.median(flux), sigma

def compare(model_phase, model_flux, obs_phase, obs_flux):
    predicted = np.interp((obs_phase+.5)%1-.5, model_phase, model_flux)
    scale = np.dot(obs_flux, predicted)/np.dot(predicted, predicted)
    residual = obs_flux-scale*predicted
    return scale*predicted, residual, float(np.sqrt(np.mean(residual**2)))

def export_native_csv(path, parameters, phases=513, triangles=1500):
    grid = np.linspace(-.5, .5, phases)
    x, y = native_flux(**parameters, phases=grid, triangles=triangles)
    np.savetxt(path, np.column_stack([x,y]), delimiter=",", header="phase,normalized_flux", comments="")
    return x, y


## Runnable native recovery benchmark

This intentionally fixes q=0.75 and uses a high-inclination synthetic system, so it demonstrates recovery of inclination, fill-out, and temperature ratio under a controlled, identifiable setup. It is not evidence that photometry alone uniquely recovers q. The search phase budget and final 513-phase validation are reported separately.

In [ ]:
RUN_NATIVE_BENCHMARK = False  # Set True when ready; expect several native PHOEBE calculations.
if RUN_NATIVE_BENCHMARK:
    truth = dict(inclination=88., mass_ratio=.75, fillout_factor=.32,
                 temperature_ratio=.90, period=1.8, irradiation=False)
    fit_grid = np.linspace(-.5, .5, 65)
    _, observed = native_flux(**truth, phases=fit_grid)
    def residual(v):
        _, model = native_flux(inclination=v[0], mass_ratio=truth["mass_ratio"],
                               fillout_factor=v[1], temperature_ratio=v[2],
                               period=1.8, phases=fit_grid)
        return model-observed
    result = least_squares(residual, [84., .22, .84], bounds=([65,.10,.78],[90,.50,1.12]), max_nfev=12)
    recovered = dict(inclination=result.x[0], mass_ratio=truth["mass_ratio"],
                     fillout_factor=result.x[1], temperature_ratio=result.x[2],
                     period=1.8, irradiation=False)
    final_grid = np.linspace(-.5,.5,513)
    _, exact_truth = native_flux(**truth, phases=final_grid, triangles=1500)
    _, exact_fit = native_flux(**recovered, phases=final_grid, triangles=1500)
    print("65-phase search:", recovered, "optimizer converged:", result.success, result.message)
    print("513-phase native validation RMS:", np.sqrt(np.mean((exact_fit-exact_truth)**2)))


## Held-out forward checks

These points exercise shallow contact, deeper contact, and unequal mass. They compare a fast surrogate forward curve to native PHOEBE; they are not parameter-recovery or uniqueness tests.

In [ ]:
RUN_HELDOUT_FORWARD = False
HELDOUT = [
    dict(name="shallow", inclination=86., mass_ratio=.82, fillout_factor=.12, temperature_ratio=.92),
    dict(name="deep", inclination=82., mass_ratio=.90, fillout_factor=.44, temperature_ratio=.88),
    dict(name="unequal", inclination=87., mass_ratio=.50, fillout_factor=.28, temperature_ratio=.95),
]
if RUN_HELDOUT_FORWARD:
    for case in HELDOUT:
        name = case.pop("name")
        xp, yp = contact_preview(**case)
        xn, yn = native_flux(**case, phases=xp)
        print(name, "normalized-flux RMS", np.sqrt(np.mean((yp-yn)**2)), "max", np.max(np.abs(yp-yn)))
        case["name"] = name


## Portability notes

- **Colab:** run the install cell on a free CPU runtime. Widget callbacks, if added, cross the Colab kernel boundary; native PHOEBE is also kernel-side.
- **JupyterLab:** install PHOEBE in the selected kernel. Modern JupyterLab includes ipywidgets support; this notebook’s plain cells work without widgets.
- **Classic Notebook:** select the environment where PHOEBE 2.5.4 is installed. The plotting and fitting cells are otherwise unchanged.
- The embedded preview is the exact frozen branch artifact, not retrained in the notebook. Training provenance and validation metrics live in `experiments/contact/REPORT.md`.
